# Solana Anomaly Detection — Training Notebook

Loads the three monthly parquet files, builds the transaction-level feature matrix,
trains Isolation Forest + Feature-Attention Transformer AE, applies conformal prediction,


**No BigQuery queries.** All input is read from `/home/ubuntu/data/chunks/`.

## Experiment Suite

| # | Section | Goal |
|---|---------|------|
| 2 | Ablation | Measures each component's contribution vs baseline |
| 3 | Sensitivity | Measures stability of baseline across hyperparameter ranges |
| 4 | Contamination | Measures baseline detection rate on synthetic anomalies |
| 5 | Interpretability | Explains what the baseline flagged and why |

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ── Reload from checkpoints ───────────────────────────────────────────────────
import os, pickle, joblib, numpy as np, pandas as pd, torch

CKPT = '/workspace/checkpoints/'

# ── Scalers + features ────────────────────────────────────────────────────────
scaler     = joblib.load(CKPT + 'scaler.pkl')
X_tx_norm  = np.load(CKPT + 'X_tx_norm.npy')
print(f'X_tx_norm: {X_tx_norm.shape}')

# ── Scores ────────────────────────────────────────────────────────────────────
if_tx_score     = np.load(CKPT + 'if_tx_score.npy')
tr_tx_score     = np.load(CKPT + 'tr_tx_score.npy')
per_feat_errors = np.load(CKPT + 'per_feat_errors.npy')
combined_tx     = np.load(CKPT + 'combined_tx_score.npy')
cal_scores      = np.load(CKPT + 'cal_scores.npy')
print(f'Scores loaded. cal_scores: {len(cal_scores)}')

# ── Wallet-level outputs ───────────────────────────────────────────────────────
wallet_scores = pl.read_parquet(CKPT + 'wallet_scores_final.parquet').to_pandas()
df_flat       = pl.read_parquet(CKPT + 'transactions_scored.parquet').to_pandas()
print(f'wallet_scores: {wallet_scores.shape}, df_flat: {df_flat.shape}')

# ── Split identifiers ─────────────────────────────────────────────────────────
with open(CKPT + 'test_wallets.pkl', 'rb') as _f:
    test_wallets = pickle.load(_f)
with open(CKPT + 'alpha.pkl', 'rb') as _f:
    ALPHA = pickle.load(_f)
print(f'test_wallets: {len(test_wallets)}, ALPHA: {ALPHA}')

test_mask_w = wallet_scores['wallet'].isin(test_wallets).to_numpy()

# ── Models ────────────────────────────────────────────────────────────────────
iforest  = joblib.load(CKPT + 'iforest.pkl')
print('iforest loaded')

_ae_ckpt = torch.load(CKPT + 'ae_model.pt', map_location='cpu')
TR_CFG   = _ae_ckpt['cfg']
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tr_model = WalletTransformerAE(n_features=X_tx_norm.shape[1], **{k: v for k, v in TR_CFG.items() if k != 'epochs'})
tr_model.load_state_dict(_ae_ckpt['state_dict'])
tr_model = tr_model.to(device).eval()
print(f'tr_model loaded on {device}')

# ── Optional: permutation results ─────────────────────────────────────────────
if os.path.exists(CKPT + 'struct_perm_df.parquet'):
    struct_perm_df      = pl.read_parquet(CKPT + 'struct_perm_df.parquet').to_pandas()
    struct_perm_summary = pl.read_parquet(CKPT + 'struct_perm_summary.parquet').to_pandas()
    print(f'struct_perm_df: {struct_perm_df.shape}')
else:
    print('struct_perm_df not found — permutation cell not yet run')

# ── Optional: decay results ───────────────────────────────────────────────────
if os.path.exists(CKPT + 'decay_df.parquet'):
    decay_df = pl.read_parquet(CKPT + 'decay_df.parquet').to_pandas()
    print(f'decay_df: {decay_df.shape}')
else:
    print('decay_df not found — decay cell not yet run')

print('\n=== Reload complete ===')


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/checkpoints/scaler.pkl'

In [ ]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Run once on a fresh instance. Safe to re-run (skips already-installed).
import sys
!{sys.executable} -m pip install -q \
    polars pyarrow lance \
    pandas numpy scipy statsmodels joblib \
    scikit-learn optuna shap \
    matplotlib seaborn tqdm lance polar \
    ipykernel
print('Done.')


In [49]:
# ── Vast.ai setup ────────────────────────────────────────────────────────────
# RAPIDS / cuML pre-installed on RAPIDS Docker template.
# On plain PyTorch image: pip install --extra-index-url=https://pypi.nvidia.com cuml-cu12==24.10.*
import os, warnings, math
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import QuantileTransformer
from sklearn.ensemble import IsolationForest as _SklearnIF
from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
torch.backends.cuda.enable_flash_sdp(False)         # incompatible with CUDA 13.0
torch.backends.cuda.enable_mem_efficient_sdp(False)  # incompatible with CUDA 13.0

# ── GPU / AMP ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device  = torch.device('cuda')
    props   = torch.cuda.get_device_properties(0)
    USE_AMP = props.major >= 8
    print(f'Device : {props.name} | {props.total_memory // 1024**3} GB VRAM')
    print(f'AMP    : {"BFloat16" if USE_AMP else "Disabled"}')
else:
    device, USE_AMP = torch.device('cpu'), False
    print('Device : CPU')

PRECISION = torch.bfloat16 if USE_AMP else torch.float32
np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

# ── cuML FIL availability ─────────────────────────────────────────────────────
try:
    from cuml.fil import ForestInference
    _HAS_FIL = True
    print('cuML FIL : ENABLED (GPU inference via ForestInference)')
except ImportError:
    _HAS_FIL = False
    print('cuML FIL : not found — sklearn CPU scoring will be used')

# ── IsolationForest wrapper ───────────────────────────────────────────────────
# Training  : always sklearn on CPU (fast — max_samples is small per tree).
# Inference : ForestInference.load_from_sklearn transfers the fitted forest to
#             GPU via cuml/treelite; predict() scores 2.9M rows in ~1–2 s.
#             Falls back to sklearn.decision_function when FIL is unavailable.
#
# Score convention: decision_function returns sklearn-equivalent scores
#   (higher = more normal). Callers negate to get anomaly scores.
#
# free_gpu() releases the FIL handle between Optuna trials to reclaim VRAM.

class IsolationForest:
    def __init__(self, n_estimators=100, max_samples=256, contamination=0.005,
                 random_state=42, n_jobs=-1, bootstrap=False, max_features=1.0):
        _cont = 0.005 if contamination == 'auto' else float(contamination)
        self._model = _SklearnIF(
            n_estimators=n_estimators, max_samples=max_samples,
            contamination=_cont, random_state=random_state,
            n_jobs=n_jobs, bootstrap=bootstrap, max_features=max_features)
        self._fil = None

    def fit(self, X):
        X = np.asarray(X, dtype=np.float32)
        self._model.fit(X)
        if _HAS_FIL:
            try:
                self._fil = ForestInference.load_from_sklearn(
                    self._model)
            except Exception as _e:
                print(f'  FIL load failed ({_e}) — falling back to CPU scoring')
        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=np.float32)
        if self._fil is not None:
            fil_scores = np.asarray(self._fil.predict(X)).ravel()
            # FIX: Negate the FIL score to match sklearn's score_samples behaviour
            return -fil_scores - self._model.offset_
        return self._model.decision_function(X)

    def free_gpu(self):
        self._fil = None
        torch.cuda.empty_cache()


    def __getstate__(self):
        # Remove the unpicklable FIL object before saving
        state = self.__dict__.copy()
        state['_fil'] = None
        return state

    def __setstate__(self, state):
        # Restore state and try to reload FIL if available
        self.__dict__.update(state)
        if _HAS_FIL:
            try:
                self._fil = ForestInference.load_from_sklearn(self._model)
            except Exception:
                pass



Device : NVIDIA L4 | 22 GB VRAM
AMP    : BFloat16
cuML FIL : ENABLED (GPU inference via ForestInference)


In [14]:
!ls /content/drive/MyDrive/Colab-Notebooks/vastaiFinal

bastafloans.ipynb		   ubuntufixmyclipboard.ipynb
checkpoints			   vastaifloansd.ipynb
engineered_features_cache.parquet  vastaifloans.ipynb
optuna_study.db


In [15]:
# ── Load engineered features ─────────────────────────────────────────────────
import os

for _p in [
    '/content/drive/MyDrive/Colab-Notebooks/vastaiFinal/engineered_features_cache.parquet',
    '/home/ubuntu/vastaiFinal/engineered_features_cache.parquet',
    '/mnt/disk2/home/ubuntu/vastaiFinal/engineered_features_cache.parquet',
]:
    if os.path.exists(_p):
        CACHE = _p
        break
else:
    raise FileNotFoundError('engineered_features_cache.parquet not found')

print(f'Loading: {CACHE}')
df_flat = pl.read_parquet(CACHE)

print(f'Loaded : {df_flat.shape[0]:,} rows, {df_flat["wallet"].n_unique():,} wallets')
print(f'Columns: {df_flat.columns}')


Loading: /content/drive/MyDrive/Colab-Notebooks/vastaiFinal/engineered_features_cache.parquet
Loaded : 2,905,990 rows, 619,160 wallets
Columns: ['signature', 'block_timestamp', 'tx_date', 'wallet', 'fee_sol', 'compute_units_consumed', 'success_flag', 'involves_sysvar_flag', 'max_cpi_depth', 'inner_instruction_count', 'dex_hop_count', 'debt_hop_count', 'instigator_net_token_profit', 'instigator_sol_delta', 'instigator_fiat_delta', 'unique_nonsigner_account_count', 'log_count', 'num_accounts', 'num_balance_changes', 'max_sol_change', 'mint_diversity', 'unique_program_count', 'hop_density', 'avg_depth_per_protocol', 'unknown_program_count', 'has_fiat_profit', 'has_token_profit', 'has_sol_profit']


In [16]:
cache_path = '/content/drive/MyDrive/Colab-Notebooks/vastaiFinal/engineered_features_cache.parquet'

pl.read_parquet(cache_path)

signature,block_timestamp,tx_date,wallet,fee_sol,compute_units_consumed,success_flag,involves_sysvar_flag,max_cpi_depth,inner_instruction_count,dex_hop_count,debt_hop_count,instigator_net_token_profit,instigator_sol_delta,instigator_fiat_delta,unique_nonsigner_account_count,log_count,num_accounts,num_balance_changes,max_sol_change,mint_diversity,unique_program_count,hop_density,avg_depth_per_protocol,unknown_program_count,has_fiat_profit,has_token_profit,has_sol_profit
str,"datetime[μs, UTC]",date,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i8
"""4EBGdMPx2wn5A1BFsYPJKnwJtM612K…",2024-09-01 09:53:15 UTC,2024-09-01,"""GBuTZfT1ULsbdknmyJGJKiauAQ6wjy…",0.000105,40966.0,1.0,0.0,2.0,11.0,0.0,0.0,0.0,-0.004184,0.0,6.0,44.0,10.0,10.0,0.004184,0.0,4.0,0.666667,0.5,4.0,0,null,0
"""xv5gXqPPvDjqycAomi1VwE16cLThaU…",2024-09-01 03:03:46 UTC,2024-09-01,"""5y5jCvdD6uTPjWbQtLj8ABQ56jCor2…",0.000017,26756.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002056,0.0,7.0,32.0,11.0,11.0,0.202039,1.0,5.0,0.714286,0.4,5.0,0,null,0
"""3R1oye8fqo7vZCWpB8DmFXJmsLDkSZ…",2024-09-01 22:24:21 UTC,2024-09-01,"""HVVpe8FEKFTA8XzFGyyiJRwn9GwzSo…",0.000015,27854.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002054,-100.0,7.0,32.0,11.0,11.0,0.002054,1.0,5.0,0.714286,0.4,5.0,0,null,0
"""3oB21Z2VmF8s6JXWXjdbU2BvSy4Z8C…",2024-09-01 19:31:12 UTC,2024-09-01,"""G1sd2FVtvwhtowoxLFoVeMcrCkYqnh…",0.000015,29195.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002054,0.0,7.0,32.0,11.0,11.0,0.002054,1.0,5.0,0.714286,0.4,5.0,0,null,0
"""3MwBVvQCKc46STAxity7GoqmtS5PCG…",2024-09-01 03:34:55 UTC,2024-09-01,"""CpHrckWuj3jfXG36bKDr3zJgjNu4D6…",0.000015,27854.0,1.0,0.0,2.0,9.0,0.0,0.0,0.0,-0.002054,0.0,7.0,32.0,11.0,11.0,0.002054,1.0,5.0,0.714286,0.4,5.0,0,null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""5q9CVRYkkHRL8VuFTvuxNtqxKDmw1x…",2025-01-31 13:13:03 UTC,2025-01-31,"""JARDTp1s5HDNUY4yZsmxEyoVm9JCsj…",0.000006,795825.0,1.0,0.0,null,0.0,0.0,0.0,19999.982422,0.0574,19999.982422,12.0,1.0,15.0,15.0,0.057406,2.0,0.0,0.0,0.0,0.0,1,1,1
"""3FZLeMhV8ZSroMzRnb1hFxZRpBfrTb…",2025-01-31 12:20:58 UTC,2025-01-31,"""JARDTp1s5HDNUY4yZsmxEyoVm9JCsj…",0.000006,815028.0,1.0,0.0,null,0.0,0.0,0.0,20005.066406,0.0574,20005.066406,12.0,1.0,15.0,15.0,0.057406,2.0,0.0,0.0,0.0,0.0,1,1,1
"""5JZiDngyVzyqr2VtQWqS3ZFbBNFMgL…",2025-01-31 13:43:46 UTC,2025-01-31,"""JARDTp1s5HDNUY4yZsmxEyoVm9JCsj…",0.00006,902441.0,1.0,0.0,null,0.0,0.0,0.0,47.844093,2.888813,0.0,13.0,1.0,17.0,17.0,2.888813,2.0,0.0,0.0,0.0,0.0,0,1,1


In [17]:
# ── Timestamp + sort ──────────────────────────────────────────────────────────
# block_timestamp is already Datetime[us, UTC] in the parquet — no conversion needed.
df_flat = df_flat.sort(['wallet', 'block_timestamp'])

# Per-tx time since wallet's previous transaction (0 for first tx)
df_flat = df_flat.with_columns(
    pl.col('block_timestamp')
    .diff()
    .over('wallet')
    .dt.total_seconds()
    .fill_null(0.0)
    .alias('delta_time')
)
print('Timestamps OK. delta_time computed.')


Timestamps OK. delta_time computed.


In [18]:
# ── EDA: Dataset characterisation ────────────────────────────────────────────
print('=== Dataset Overview ===')
print(f'Date range : {df_flat["block_timestamp"].min()} -> {df_flat["block_timestamp"].max()}')
print(f'Rows       : {df_flat.shape[0]:,}')
print(f'Wallets    : {df_flat["wallet"].n_unique():,}')

# Transactions per wallet
_tpw = df_flat.group_by('wallet').len().rename({'len': 'n_tx'})
print('\nTransactions per wallet:')
print(_tpw['n_tx'].to_pandas().describe(percentiles=[.25, .5, .75, .90, .99]).round(1))

# Add _month as 'YYYY-MM' string (also used in rolling-window cell)
df_flat = df_flat.with_columns(
    pl.col('block_timestamp').dt.strftime('%Y-%m').alias('_month')
)

# Monthly breakdown
monthly_vol = (
    df_flat.group_by('_month')
    .agg([
        pl.len().alias('n_tx'),
        pl.col('wallet').n_unique().alias('n_wallets'),
    ])
    .sort('_month')
)

first_seen_month = (
    df_flat.group_by('wallet')
    .agg(pl.col('block_timestamp').min().dt.strftime('%Y-%m').alias('_month'))
    .group_by('_month')
    .len()
    .rename({'len': 'n_new_wallets'})
)

monthly_vol = (
    monthly_vol
    .join(first_seen_month, on='_month', how='left')
    .with_columns(pl.col('n_new_wallets').fill_null(0))
    .with_columns([
        (pl.col('n_tx') / pl.col('n_wallets').clip(lower_bound=1))
            .cast(pl.Float64).alias('tx_per_wallet'),
        (100.0 * pl.col('n_new_wallets') / pl.col('n_wallets').clip(lower_bound=1))
            .cast(pl.Float64).alias('pct_new_wallets'),
    ])
)

print('\nMonthly breakdown (expanded):')
print(monthly_vol.to_pandas().to_string(index=False))

single_tx = (_tpw['n_tx'] == 1).sum()
multi_tx  = (_tpw['n_tx'] > 1).sum()
print(f'\nSingle-tx wallets: {single_tx:,}  ({single_tx / len(_tpw) * 100:.1f}%)')
print(f'Multi-tx wallets : {multi_tx:,}')

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

# A) Wallet activity concentration
_tpw_np = _tpw['n_tx'].to_numpy()
sns.histplot(_tpw_np, bins=60, log_scale=(True, False), color='#4C78A8', ax=axes[0])
axes[0].set_title('Transactions per Wallet (log-x)')
axes[0].set_xlabel('Transactions per wallet')
axes[0].set_ylabel('Wallet count')

# B) Monthly turnover and wallet freshness
_month_plot = monthly_vol.to_pandas()
sns.barplot(data=_month_plot, x='_month', y='tx_per_wallet', color='#F58518', ax=axes[1])
axes[1].set_title('Tx per Active Wallet by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Transactions per active wallet')
ax1b = axes[1].twinx()
sns.lineplot(data=_month_plot, x='_month', y='pct_new_wallets',
             marker='o', color='#54A24B', linewidth=1.8, ax=ax1b)
ax1b.set_ylabel('New-wallet share (%)')
ax1b.grid(False)

# C) Key feature distributions
# (instigator_sol_delta + instigator_net_token_profit replace the former
#  balance_churn_rate / total_vol which are not in this dataset)
key_feats = ['fee_sol', 'max_sol_change', 'instigator_sol_delta', 'instigator_net_token_profit']
_dist_parts = []
for feat in key_feats:
    vals = df_flat[feat].cast(pl.Float64).to_numpy()
    lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
    _dist_parts.append(pd.DataFrame({'feature': feat, 'value': np.clip(vals, lo, hi)}))

dist_df = pd.concat(_dist_parts, ignore_index=True)
sns.boxenplot(data=dist_df, x='feature', y='value', ax=axes[2], color='#72B7B2')
axes[2].set_title('Key Feature Spread (clipped [1%, 99%])')
axes[2].set_xlabel('Feature')
axes[2].set_ylabel('Value')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: eda_overview.png')


=== Dataset Overview ===
Date range : 2024-09-01 00:00:02+00:00 -> 2025-01-31 23:40:43+00:00
Rows       : 2,905,990
Wallets    : 619,160

Transactions per wallet:
count    619160.0
mean          4.7
std         367.6
min           1.0
25%           1.0
50%           1.0
75%           1.0
90%           1.0
99%          48.0
max      178862.0
Name: n_tx, dtype: float64

Monthly breakdown (expanded):
 _month   n_tx  n_wallets  n_new_wallets  tx_per_wallet  pct_new_wallets
2024-09 500791     122696         122696       4.081559       100.000000
2024-10 628683     154409         152071       4.071544        98.485840
2024-11 760275     176601         172775       4.305044        97.833534
2024-12 707659     131707         125870       5.372979        95.568193
2025-01 308582      50606          45748       6.097735        90.400348

Single-tx wallets: 579,823  (93.6%)
Multi-tx wallets : 39,337
Saved: eda_overview.png


In [19]:
# ── Feature columns ───────────────────────────────────────────────────────────
# Source parquet: 28 cols  →  30 in df_flat (+ delta_time, _month)
# cs.numeric() picks up all 25 usable features automatically:
#   21 × Float32  original transaction metrics
#    3 × Int8     has_fiat_profit / has_token_profit / has_sol_profit
#    1 × Float64  delta_time  (inter-tx interval, added in cell 5)
# Excluded (5): signature, block_timestamp, tx_date, wallet, _month
tx_feature_cols = df_flat.select(cs.numeric()).columns  # list[str], length 25

# Cast everything to Float32, replace inf / nan → 0
df_flat = df_flat.with_columns([
    pl.col(col).cast(pl.Float32).fill_nan(0.0).fill_null(0.0)
    for col in tx_feature_cols
])

X_tx = df_flat.select(tx_feature_cols).to_numpy().astype(np.float32)  # (N_tx, 25)
n_inf = np.isinf(X_tx).sum()
n_nan = np.isnan(X_tx).sum()
print(f'Features      : {tx_feature_cols}')
print(f'X_tx shape    : {X_tx.shape}  |  inf={n_inf}  nan={n_nan}')
assert n_inf == 0 and n_nan == 0, "Unexpected inf/nan in feature matrix!"


Features      : ['fee_sol', 'compute_units_consumed', 'success_flag', 'involves_sysvar_flag', 'max_cpi_depth', 'inner_instruction_count', 'dex_hop_count', 'debt_hop_count', 'instigator_net_token_profit', 'instigator_sol_delta', 'instigator_fiat_delta', 'unique_nonsigner_account_count', 'log_count', 'num_accounts', 'num_balance_changes', 'max_sol_change', 'mint_diversity', 'unique_program_count', 'hop_density', 'avg_depth_per_protocol', 'unknown_program_count', 'has_fiat_profit', 'has_token_profit', 'has_sol_profit', 'delta_time']
X_tx shape    : (2905990, 25)  |  inf=0  nan=0


In [20]:
# ── Rolling Window Split Logic ────────────────────────────────────────────────
# _month is a 'YYYY-MM' string column added in the EDA cell.
all_months = sorted(df_flat['_month'].unique().to_list())
print(f'Available months in dataset: {all_months}')

if len(all_months) < 3:
    raise ValueError(f"Need at least 3 months for a rolling window, found {len(all_months)}: {all_months}")

rolling_windows = []
for i in range(len(all_months) - 2):
    rolling_windows.append((all_months[i], all_months[i+1], all_months[i+2]))

print(f'Defined {len(rolling_windows)} rolling windows:')
for i, w in enumerate(rolling_windows):
    print(f'  W{{i+1}}: Train={{w[0]}}, Val={{w[1]}}, Test={{w[2]}}')

def get_window_masks(df, train_m, val_m, test_m):
    train_w = set(df.filter(pl.col('_month') == train_m)['wallet'].unique().to_list())
    val_w   = set(df.filter(pl.col('_month') == val_m  )['wallet'].unique().to_list())
    test_w  = set(df.filter(pl.col('_month') == test_m )['wallet'].unique().to_list())

    # Ensure no overlap for strict evaluation
    val_w  = val_w  - train_w
    test_w = test_w - train_w - val_w

    m_train = (df['wallet'].is_in(list(train_w)) & (df['_month'] == train_m)).to_numpy()
    m_val   = (df['wallet'].is_in(list(val_w))   & (df['_month'] == val_m  )).to_numpy()
    m_test  = (df['wallet'].is_in(list(test_w))  & (df['_month'] == test_m )).to_numpy()

    return (m_train, m_val, m_test,
            np.array(list(train_w)), np.array(list(val_w)), np.array(list(test_w)))

# Initialize masks for the first window
train_m, val_m, test_m = rolling_windows[0]
train_mask_tx, val_mask_tx, test_mask_tx, train_wallets, val_wallets, test_wallets = \
    get_window_masks(df_flat, train_m, val_m, test_m)


Available months in dataset: ['2024-09', '2024-10', '2024-11', '2024-12', '2025-01']
Defined 3 rolling windows:
  W{i+1}: Train={w[0]}, Val={w[1]}, Test={w[2]}
  W{i+1}: Train={w[0]}, Val={w[1]}, Test={w[2]}
  W{i+1}: Train={w[0]}, Val={w[1]}, Test={w[2]}


In [21]:
# ── Normalisation (fit on train only) ─────────────────────────────────────────
# FIX: Replaced RobustScaler+clipping with QuantileTransformer.
# This forces extreme long-tailed economic data and bursty counts into a
# smooth uniform [0, 1] distribution, completely avoiding gradient explosion
# while preserving the rank-extremity of outliers without blind clipping.

from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=42)
scaler.fit(X_tx[train_mask_tx])
X_tx_norm = scaler.transform(X_tx).astype(np.float32)

n_inf = np.isinf(X_tx_norm).sum()
n_nan = np.isnan(X_tx_norm).sum()
print(f'Normalised  |  inf={n_inf}  nan={n_nan}')
assert n_inf == 0 and n_nan == 0, "inf/nan after normalisation — check raw features!"

print('')
print(pd.DataFrame(X_tx_norm[train_mask_tx], columns=tx_feature_cols)
      .describe().loc[['mean','50%','std']].round(3))


Normalised  |  inf=0  nan=0

      fee_sol  compute_units_consumed  success_flag  involves_sysvar_flag  \
mean   -0.790                   0.003        -5.199                -4.939   
50%     0.027                   0.013        -5.199                -5.199   
std     2.371                   0.997         0.027                 1.617   

      max_cpi_depth  inner_instruction_count  dex_hop_count  debt_hop_count  \
mean         -0.069                    0.000         -0.332          -5.198   
50%          -0.388                   -0.268         -0.034          -5.199   
std           1.112                    0.973          1.670           0.121   

      instigator_net_token_profit  instigator_sol_delta  ...  max_sol_change  \
mean                       -0.400                 0.010  ...           0.004   
50%                        -0.006                 0.008  ...           0.048   
std                         1.879                 0.999  ...           1.008   

      mint_diversity  un

In [22]:
!pip install lancedb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 37.5 MB/s eta 0:00:00


In [25]:
# ── Write Lance dataset ───────────────────────────────────────────────────────
# One row per transaction: wallet + _month + 25 normalised features.
# Uses lancedb (high-level API) for writing — compatible across lance versions.
# WalletSequenceDataset opens the dataset via lance.dataset() for low-level reads.
import lancedb, pyarrow as pa
import os
import numpy as np
import pandas as pd
import polars as pl
import lance

for _base in ['.', '/home/ubuntu/vastaiFinal', '/mnt/disk2/home/ubuntu/vastaiFinal']:
    if os.path.isdir(_base):
        LANCE_DIR   = _base
        LANCE_TABLE = 'transactions_norm'
        LANCE_PATH  = os.path.join(_base, LANCE_TABLE + '.lance')
        break

_arrow = {
    'wallet': pa.array(df_flat['wallet']),
    '_month': pa.array(df_flat['_month']),
}
for i, col in enumerate(tx_feature_cols):
    _arrow[col] = pa.array(X_tx_norm[:, i])

_db  = lancedb.connect(LANCE_DIR)
_tbl = _db.create_table(LANCE_TABLE, pa.table(_arrow), mode='overwrite')
print(f'Lance written : {LANCE_PATH}')
print(f'Rows: {_tbl.count_rows():,}   Cols: {len(_tbl.schema)}')

# Build wallet → (start_row, seq_len) index via polars groupby
# Handle both Pandas and Polars df_flat automatically
if isinstance(df_flat, pd.DataFrame):
    _df_subset = pl.from_pandas(df_flat[['wallet']])
else:
    _df_subset = df_flat.select('wallet')

_idx_df = (
    _df_subset
    .with_row_index('_row')
    .group_by('wallet')
    .agg([pl.col('_row').min().alias('start_row'), pl.len().alias('seq_len')])
)

WALLET_INDEX = {
    row[0]: (int(row[1]), int(row[2]))
    for row in _idx_df.iter_rows()
}
print(f'Wallet index  : {len(WALLET_INDEX):,} wallets')

def sample_lance_data(lance_path, n_samples, columns=None, seed=42):
    """High-performance random sampling using Lance scanner."""
    ds = lance.dataset(lance_path)

Lance written : ./transactions_norm.lance
Rows: 2,905,990   Cols: 27
Wallet index  : 619,160 wallets


In [ ]:
# ── Checkpoint: save scaler + normalised features ────────────────────────────
import os, joblib, numpy as np

CKPT = '/workspace/checkpoints/'
os.makedirs(CKPT, exist_ok=True)

joblib.dump(scaler, CKPT + 'scaler.pkl')
print('Saved: scaler.pkl')

np.save(CKPT + 'X_tx_norm.npy', X_tx_norm)
print('Saved: X_tx_norm.npy')


In [26]:
# ── Feature collinearity & VIF analysis ──────────────────────────────────────
import warnings

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    _HAS_SM = True

except ImportError:
    _HAS_SM = False
    print('statsmodels not found — skipping VIF (pip install statsmodels to enable).')

# Pearson correlation on train split (normalised)
_corr = pl.from_pandas(pd.DataFrame(X_tx_norm[train_mask_tx], columns=tx_feature_cols)).corr().to_pandas()
_corr.index = tx_feature_cols

# High-correlation pairs (|r| > 0.80)
_hc = []
for _i in range(len(tx_feature_cols)):
    for _j in range(_i + 1, len(tx_feature_cols)):
        _r = _corr.iloc[_i, _j]
        if abs(_r) > 0.80:
            _hc.append({'feat_a': tx_feature_cols[_i],
                        'feat_b': tx_feature_cols[_j],
                        'pearson_r': round(float(_r), 3)})
hc_df = (pd.DataFrame(_hc)
         .sort_values('pearson_r', ascending=False, key=abs)
         .reset_index(drop=True))
print('Highly correlated pairs (|r| > 0.80):')
print(hc_df.to_string(index=False) if len(hc_df) else '  None found.')

# VIF
if _HAS_SM:
    _rng0 = np.random.default_rng(42)
    _n    = train_mask_tx.sum()
    _samp = _rng0.choice(_n, size=min(8000, _n), replace=False)
    _Xv   = X_tx_norm[train_mask_tx][_samp]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _vif = [variance_inflation_factor(_Xv, _k) for _k in range(_Xv.shape[1])]
    vif_df = (pd.DataFrame({'feature': tx_feature_cols, 'VIF': _vif})
              .sort_values('VIF', ascending=False))
    print('\nVariance Inflation Factors (top 15):')
    print(vif_df.head(15).to_string(index=False))
    print(f'Features with VIF > 10: {vif_df[vif_df.VIF > 10]["feature"].tolist()}')

# Heatmap (Seaborn)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    _corr,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    xticklabels=tx_feature_cols,
    yticklabels=tx_feature_cols,
    cbar_kws={'shrink': 0.75, 'label': 'Pearson r'},
    ax=ax,
)
ax.set_title('Feature Pearson Correlation Matrix (train split)', fontsize=11)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)
plt.tight_layout()
plt.savefig('feature_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: feature_correlation_heatmap.png')


Highly correlated pairs (|r| > 0.80):
                        feat_a                feat_b  pearson_r
                  num_accounts   num_balance_changes      1.000
unique_nonsigner_account_count          num_accounts      0.957
unique_nonsigner_account_count   num_balance_changes      0.957
       inner_instruction_count             log_count      0.954
   instigator_net_token_profit      has_token_profit      0.907
          unique_program_count unknown_program_count      0.877
         instigator_fiat_delta       has_fiat_profit      0.864

Variance Inflation Factors (top 15):
                       feature          VIF
           num_balance_changes 2.933941e+13
                  num_accounts 2.933941e+13
unique_nonsigner_account_count 4.214102e+01
          unique_program_count 3.849728e+01
         unknown_program_count 2.842177e+01
                 max_cpi_depth 2.400882e+01
        avg_depth_per_protocol 1.795655e+01
       inner_instruction_count 1.398367e+01
                

In [27]:
# ── Isolation Forest ──────────────────────────────────────────────────────────
# Trains on ~546k raw transaction rows.
# max_samples=256 keeps each tree tractable regardless of dataset size.
#from sklearn.preprocessing import RobustScaler
# contamination=0.02: cascade-filtered dataset has ~3-10% attack rate;
# 'auto' (=0.1 sklearn default) is too permissive; 0.005 too conservative.
IF_CFG = dict(n_estimators=300, max_samples=256,
              contamination=0.02, random_state=42, n_jobs=-1)

print('Fitting Isolation Forest...')
iforest = IsolationForest(**IF_CFG).fit(X_tx[train_mask_tx])  # FIX: Tree models are scale-invariant; feed raw data

if_raw      = -iforest.decision_function(X_tx)           # (N_tx,) higher = more anomalous

# Training-CDF normalisation (Kriegel et al. 2011 "Interpreting and Unifying
# Outlier Scores"; ECOD, Li et al. 2022):
#   score_i = P(X_train <= raw_i)  ≈ empirical CDF of the training distribution.
# Using training-only CDF avoids data leakage: val/test raw scores do NOT
# influence the normalisation reference.  Mean ≈ 0.50 on training data by
# construction; val/test anomalies push into the upper tail (>0.95).
if_train_raw   = if_raw[train_mask_tx]
_if_train_sort = np.sort(if_train_raw)
# Empirical CDF via searchsorted — O(N log N), equivalent to loop but ~1000x faster.
if_tx_score    = np.searchsorted(_if_train_sort, if_raw, side='right') / len(_if_train_sort)  # (N_tx,) ∈ [0,1]
print(f'IF done. Score range : [{if_tx_score.min():.4f}, {if_tx_score.max():.4f}]')
print(f'IF train mean (≈0.50 by construction) : {if_tx_score[train_mask_tx].mean():.4f}')
print(f'IF all-tx mean (>0.50 expected — val/test anomalies): {if_tx_score.mean():.4f}')

Fitting Isolation Forest...
IF done. Score range : [0.0004, 1.0000]
IF train mean (≈0.50 by construction) : 0.5130
IF all-tx mean (>0.50 expected — val/test anomalies): 0.6684


In [41]:
# ── WalletSequenceDataset ─────────────────────────────────────────────────────
# Each wallet = one sequence of up to MAX_SEQ_LEN transactions (most-recent-last,
# left-padded with zeros). pad_mask is True for padding (PyTorch convention).
#
# OPTIMIZED: Now supports Lance-backed out-of-core loading. If X is a string
# (path) or a Lance dataset object, rows are pulled from disk only when needed.

import lance
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 64   # p99 wallet tx count ≈ 48; 64 gives headroom

class WalletSequenceDataset(Dataset):
    """
    Returns (seq, pad_mask) per wallet.
      seq      : (MAX_SEQ_LEN, F) float32
      pad_mask : (MAX_SEQ_LEN,)  bool — True = padding
    """
    def __init__(self, X, wallet_index, wallet_list, feature_cols, max_seq_len=MAX_SEQ_LEN):
        self.index   = wallet_index                               # {wallet: (start, len)}
        self.wallets = [w for w in wallet_list if w in wallet_index]
        self.T       = max_seq_len
        self.feature_cols = feature_cols
        self.F       = len(feature_cols)

        # Robust check: X is Lance if it's a path string or a Lance-specific object
        is_lance_obj = 'lance' in str(type(X)).lower()
        if isinstance(X, str) or is_lance_obj:
            self.ds = X if not isinstance(X, str) else lance.dataset(X)
            self.X  = None
        else:
            self.X  = X        # (N_tx, F) float32 in-memory
            self.ds = None

    def __len__(self):
        return len(self.wallets)

    def __getitem__(self, idx):
        w            = self.wallets[idx]
        start, total = self.index[w]
        T_actual     = min(total, self.T)
        row_start    = start + (total - T_actual)

        if self.ds is not None:
            # Lance disk-based retrieval
            seq_np = self.ds.take(range(row_start, row_start + T_actual),
                                  columns=self.feature_cols).to_pandas().values.astype(np.float32)
        else:
            # In-memory NumPy retrieval
            seq_np = self.X[row_start : row_start + T_actual]

        seq  = np.zeros((self.T, self.F), dtype=np.float32)
        mask = np.ones (self.T,            dtype=bool)
        seq [-T_actual:] = seq_np
        mask[-T_actual:] = False

        return torch.from_numpy(seq.copy()), torch.from_numpy(mask)


def make_wallet_loader(X, wallet_index, wallet_list, feature_cols,
                       max_seq_len=MAX_SEQ_LEN, batch_size=512,
                       shuffle=True, num_workers=0):
  ds = WalletSequenceDataset(X, wallet_index, wallet_list, feature_cols, max_seq_len)
  return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          pin_memory=torch.cuda.is_available(),
                          num_workers=num_workers)

In [45]:
# ── Wallet Sequential Transformer AE (Masked) ────────────────────────────────
# Masked Autoencoder: randomly replace 15% of real input tokens with a learnable
# [MASK] embedding during training; compute loss ONLY on those positions.
# Forces contextual learning — the model cannot copy tokens; it must infer masked
# transactions from neighbours.  At inference, no masking: reconstruction error
# on all real tokens is the anomaly signal.

class WalletTransformerAE(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4,
                  num_layers=2, ff_dim=128, dropout=0.1, max_seq_len=64,
                  mask_ratio=0.15):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_embed  = nn.Embedding(max_seq_len, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mask_ratio = mask_ratio
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation='gelu')
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, n_features)

    def forward(self, x, src_key_padding_mask=None, token_mask=None):
        # x          : (B, T, F)
        # token_mask : (B, T) bool — True = replace with mask_token (training only)
        B, T, F = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.input_proj(x) + self.pos_embed(pos).unsqueeze(0)  # (B, T, d_model)
        if token_mask is not None:
            tok = torch.where(token_mask.unsqueeze(-1),
                              self.mask_token.expand(B, T, -1), tok)
        z = self.encoder(tok, src_key_padding_mask=src_key_padding_mask)
        return self.out_proj(z)   # (B, T, F)


def _masked_mse(pred, target, compute_mask):
    """MSE on positions where compute_mask=True — optimal for N(0,1) targets."""
    err    = ((pred - target) ** 2).mean(dim=-1)   # (B, T)
    err    = err.masked_fill(~compute_mask, 0.0)
    n_real = compute_mask.float().sum().clamp(min=1)
    return err.sum() / n_real


TR_CFG = dict(d_model=64, nhead=4, num_layers=2, ff_dim=128, dropout=0.10,
              lr=1e-3, weight_decay=1e-4, epochs=30, batch_size=512, mask_ratio=0.15)

import copy

# 1. Updated Trainer (with reporting and pruning support)
def train_wallet_ae(train_loader, val_loader, cfg, n_features, max_seq_len, device, trial=None):
    model = WalletTransformerAE(
        n_features, cfg['d_model'], cfg['nhead'],
        cfg['num_layers'], cfg['ff_dim'], cfg['dropout'],
        max_seq_len=max_seq_len,
        mask_ratio=cfg.get('mask_ratio', 0.15)
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    best_val, patience_count = float('inf'), 0
    patience, min_delta = 3, 1e-4
    best_weights = copy.deepcopy(model.state_dict())
    train_losses, val_losses = [], []

    for epoch in range(cfg['epochs']):
        model.train()
        total = 0
        for seq, pad_mask in train_loader:
            seq, pad_mask = seq.to(device), pad_mask.to(device)
            rand_mask = (torch.rand_like(pad_mask, dtype=torch.float) < model.mask_ratio) & ~pad_mask
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=PRECISION):
                pred = model(seq, src_key_padding_mask=pad_mask, token_mask=rand_mask)
                loss = _masked_mse(pred, seq, rand_mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()

        train_loss = total / len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        vtot = 0
        with torch.no_grad():
            for seq, pad_mask in val_loader:
                seq, pad_mask = seq.to(device), pad_mask.to(device)
                rand_mask = (torch.rand_like(pad_mask, dtype=torch.float) < model.mask_ratio) & ~pad_mask
                with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=PRECISION):
                    pred = model(seq, src_key_padding_mask=pad_mask, token_mask=rand_mask)
                    vtot += _masked_mse(pred, seq, rand_mask).item()

        val_loss = vtot / max(len(val_loader), 1)
        val_losses.append(val_loss)
        print(f'  Epoch {epoch+1}/{cfg["epochs"]}  train={train_loss:.6f}  val={val_loss:.6f}')

        # Optuna Pruning
        if trial is not None:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        if val_loss < best_val - min_delta:
            best_val, patience_count = val_loss, 0
            best_weights = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f'  Early stop at epoch {epoch+1}. Restoring best weights.')
                break

    model.load_state_dict(best_weights)
    return model, train_losses, val_losses



def score_wallet_ae(model, X, wallet_index, all_wallets,
                    max_seq_len, device, feature_cols, batch_size=512):
    """
    Clean forward pass (no masking) — measures how well the model reconstructs
    each token from context.  High error = token deviates from learned normal pattern.
    Returns:
      wallet_scores : (N_wallets,) mean MSE over real tokens
      per_tx_matrix : (N_wallets, MAX_SEQ_LEN) per-token MSE, nan for padding
    """
    loader = make_wallet_loader(X, wallet_index, all_wallets, feature_cols,
                                max_seq_len, batch_size=batch_size, shuffle=False)
    model.eval()
    w_scores, tx_mat = [], []

    with torch.no_grad():
        for seq, mask in loader:
            seq, mask = seq.to(device), mask.to(device)
            pred      = model(seq, src_key_padding_mask=mask)   # no token_mask
            fe        = ((pred - seq) ** 2).mean(dim=-1)        # (B, T)
            fe        = fe.masked_fill(mask, float('nan'))
            real_cnt  = (~mask).float().sum(dim=1).clamp(min=1)
            w_scores.append(fe.nan_to_num(0.0).sum(dim=1).div(real_cnt).cpu().numpy())
            tx_mat.append(fe.cpu().numpy())

    return np.concatenate(w_scores), np.concatenate(tx_mat)


# ── Build DataLoaders ─────────────────────────────────────────────────────────
print('Building wallet DataLoaders...')
train_loader = make_wallet_loader(X_tx_norm, WALLET_INDEX, train_wallets.tolist(), tx_feature_cols,
                                  MAX_SEQ_LEN, batch_size=TR_CFG['batch_size'])
val_loader   = make_wallet_loader(X_tx_norm, WALLET_INDEX, val_wallets.tolist(), tx_feature_cols,
                                  MAX_SEQ_LEN, batch_size=TR_CFG['batch_size'], shuffle=False)
print(f'Train: {len(train_loader.dataset):,} wallets   Val: {len(val_loader.dataset):,} wallets')

# ── Train ─────────────────────────────────────────────────────────────────────
print('\nTraining WalletTransformerAE...')
tr_model, ae_train_losses, ae_val_losses = train_wallet_ae(
    train_loader, val_loader, TR_CFG,
    n_features=len(tx_feature_cols), max_seq_len=MAX_SEQ_LEN, device=device)

# ── Score all wallets ─────────────────────────────────────────────────────────
print('\nScoring all wallets...')
_all_wallets = df_flat['wallet'].unique().to_list()
tr_wallet_raw, per_feat_errors = score_wallet_ae(
    tr_model, X_tx_norm, WALLET_INDEX, _all_wallets, MAX_SEQ_LEN, device, tx_feature_cols)

# CDF normalisation — reference built on training wallets only (no leakage)
_train_w_mask = np.isin(np.array(_all_wallets), np.array(train_wallets))
_sorted       = np.sort(tr_wallet_raw[_train_w_mask])
tr_tx_score   = np.searchsorted(_sorted, tr_wallet_raw, side='right') / len(_sorted)
print(f'TR done. Score range: [{tr_tx_score.min():.4f}, {tr_tx_score.max():.4f}]')
print(f'TR train mean (≈0.50 by construction): {tr_tx_score[_train_w_mask].mean():.4f}')
print(f'TR all-wallet mean (>0.50 expected):   {tr_tx_score.mean():.4f}')

Building wallet DataLoaders...
Train: 122,696 wallets   Val: 152,071 wallets

Training WalletTransformerAE...
  Epoch 1/30  train=1.386246  val=0.760905
  Epoch 2/30  train=0.912126  val=0.742153
  Epoch 3/30  train=0.889250  val=0.742399
  Epoch 4/30  train=0.886636  val=0.728383
  Epoch 5/30  train=0.855087  val=0.731427
  Epoch 6/30  train=0.857440  val=0.736329
  Epoch 7/30  train=0.848876  val=0.719605
  Epoch 8/30  train=0.846392  val=0.720395
  Epoch 9/30  train=0.852154  val=0.708908
  Epoch 10/30  train=0.858059  val=0.736904
  Epoch 11/30  train=0.837520  val=0.725768
  Epoch 12/30  train=0.836628  val=0.715973
  Early stop at epoch 12. Restoring best weights.

Scoring all wallets...
TR done. Score range: [0.0000, 1.0000]
TR train mean (≈0.50 by construction): 0.5000
TR all-wallet mean (>0.50 expected):   0.4506


### AE model selection

Per the **universal workflow of ML** (Chollet, *Deep Learning with Python*): after beating a baseline, we should *regularise and tune* by searching hyperparameters rather than comparing only a few hand-picked configs. The section below first runs a compact **capacity + regularised diagnostics** pass (small/medium/large plus regularised variants), then a full **Optuna optimisation** study over AE and IF settings.

In [ ]:
# ── Capacity + regularised diagnostics (pre-Optuna) ──────────────────────────
# Sanity check: compare model capacity and regularisation before
# full hyperparameter optimisation. Uses wallet-level training/scoring.

if 'train_wallet_ae' not in globals() or 'score_wallet_ae' not in globals() or 'TR_CFG' not in globals():
    raise RuntimeError('Run AE setup first (`train_wallet_ae`, `score_wallet_ae`, `TR_CFG` required).')

if 'X_tx_norm' not in globals() or 'train_wallets' not in globals() or 'val_wallets' not in globals():
    raise RuntimeError('Missing data splits (`X_tx_norm`, `train_wallets`, `val_wallets`).')

DIAG_EPOCHS     = int(min(TR_CFG.get('epochs', 10), 6))
DIAG_BATCH_SIZE = int(TR_CFG.get('batch_size', 512))
ALPHA_DIAG      = float(ALPHA) if 'ALPHA' in globals() else 0.01

_base_lr      = float(TR_CFG.get('lr', 1e-3))
_base_wd      = float(TR_CFG.get('weight_decay', 1e-4))
_base_dropout = float(TR_CFG.get('dropout', 0.10))
_mask_ratio   = float(TR_CFG.get('mask_ratio', 0.15))

_diag_cfgs = [
    dict(profile='small', d_model=32, nhead=4, num_layers=1, ff_dim=64,
          dropout=0.10, lr=_base_lr, weight_decay=_base_wd, mask_ratio=_mask_ratio),
    dict(profile='medium', d_model=64, nhead=4, num_layers=2, ff_dim=128,
          dropout=max(0.10, _base_dropout), lr=_base_lr, weight_decay=_base_wd, mask_ratio=_mask_ratio),
    dict(profile='large', d_model=128, nhead=8, num_layers=3, ff_dim=256,
          dropout=0.15, lr=_base_lr, weight_decay=_base_wd, mask_ratio=_mask_ratio),
    dict(profile='medium_regularised', d_model=64, nhead=4, num_layers=2, ff_dim=128,
          dropout=max(0.20, _base_dropout), lr=_base_lr,
          weight_decay=max(_base_wd * 5.0, 1e-4), mask_ratio=_mask_ratio),
    dict(profile='large_regularised', d_model=128, nhead=8, num_layers=3, ff_dim=256,
          dropout=0.25, lr=_base_lr * 0.7,
          weight_decay=max(_base_wd * 8.0, 2e-4), mask_ratio=_mask_ratio),
]

# Build wallet arrays once — np.isin is O(N+M), no per-row set construction
_all_w_diag        = df_flat['wallet'].unique().to_list()
_train_w_list      = train_wallets.tolist() if hasattr(train_wallets, 'tolist') else list(train_wallets)
_val_w_list_diag   = val_wallets.tolist()   if hasattr(val_wallets,   'tolist') else list(val_wallets)
_train_w_mask_diag = np.isin(_all_w_diag, _train_w_list)
_val_w_mask_diag   = np.isin(_all_w_diag, _val_w_list_diag)

_diag_rows = []
for _cfg_base in _diag_cfgs:
    _cfg = dict(_cfg_base)
    _cfg['epochs'] = DIAG_EPOCHS
    # Reduce batch size for large models to avoid OOM
    _diag_bs = DIAG_BATCH_SIZE if _cfg.get('d_model', 64) <= 64 else max(DIAG_BATCH_SIZE // 2, 256)
    _cfg['batch_size'] = _diag_bs

    print(
        f"Diagnostic [{_cfg['profile']}]: d={_cfg['d_model']} h={_cfg['nhead']} "
        f"L={_cfg['num_layers']} ff={_cfg['ff_dim']} dr={_cfg['dropout']:.2f} "
        f"lr={_cfg['lr']:.2e} wd={_cfg['weight_decay']:.2e}"
    )

    _tr_loader_diag  = make_wallet_loader(X_tx_norm, WALLET_INDEX, _train_w_list, tx_feature_cols,
                                          MAX_SEQ_LEN, batch_size=_diag_bs)
    _val_loader_diag = make_wallet_loader(X_tx_norm, WALLET_INDEX, _val_w_list_diag, tx_feature_cols,
                                          MAX_SEQ_LEN, batch_size=_diag_bs, shuffle=False)

    _m, _train_losses, _val_losses = train_wallet_ae(
        _tr_loader_diag, _val_loader_diag, _cfg,
        n_features=len(tx_feature_cols), max_seq_len=MAX_SEQ_LEN, device=device)

    _raw_diag, _ = score_wallet_ae(_m, X_tx_norm, WALLET_INDEX, _all_w_diag, MAX_SEQ_LEN, device, tx_feature_cols)
    # CDF reference: training wallets only (no leakage into val/test)
    _diag_sort  = np.sort(_raw_diag[_train_w_mask_diag])
    _score_diag = np.searchsorted(_diag_sort, _raw_diag, side='right') / len(_diag_sort)
    _val_sc     = _score_diag[_val_w_mask_diag]

    _tail = float(np.quantile(_val_sc, 0.99) - np.quantile(_val_sc, 0.50))
    _thr  = float(np.quantile(_val_sc, 1 - ALPHA_DIAG))

    del _m
    torch.cuda.empty_cache()

    _diag_rows.append({
        'profile':            _cfg['profile'],
        'regularised':        ('regularised' in _cfg['profile']),
        'train_mse':          float(_train_losses[-1]),
        'val_mse':            float(_val_losses[-1]),
        'generalisation_gap': float(_val_losses[-1] - _train_losses[-1]),
        'val_tail_contrast':  _tail,
        'val_threshold':      _thr,
        'd_model':            int(_cfg['d_model']),
        'nhead':              int(_cfg['nhead']),
        'num_layers':         int(_cfg['num_layers']),
        'dropout':            float(_cfg['dropout']),
        'lr':                 float(_cfg['lr']),
        'weight_decay':       float(_cfg['weight_decay']),
        'epochs':             int(_cfg['epochs']),
        'batch_size':         int(_cfg['batch_size']),
    })

chollet_capacity_regularised_df = (
    pd.DataFrame(_diag_rows)
    .sort_values(['val_mse', 'generalisation_gap', 'val_tail_contrast'],
                  ascending=[True, True, False])
    .reset_index(drop=True)
)

print('\nCapacity + regularised diagnostics:')
print(chollet_capacity_regularised_df.to_string(index=False))

_best_profile = chollet_capacity_regularised_df.iloc[0]['profile']
print(f'\nRecommended centre for Optuna optimisation: {_best_profile}')
print('Saved table: `capacity_regularised_df`.')

In [44]:
import optuna, os

_study_db  = 'sqlite:////content/drive/MyDrive/Colab-Notebooks/vastaiFinal/optuna_study.db'
_study_name = 'vastaifloans'

try:
    optuna.delete_study(study_name=_study_name, storage=_study_db)
    print(f'Study "{_study_name}" deleted — will start fresh from trial 0.')
except KeyError:
    print('Study not found — nothing to delete.')

Study "vastaifloans" deleted — will start fresh from trial 0.


In [ ]:
import os
print(os.cpu_count())

In [ ]:
"""
import torch, gc
# 1. Clear known global model variables if they exist
for var in ['iforest', 'tr_model', 'if_model', '_m', 'if_win']:
  if var in globals():
  # If it's your wrapper, call its internal free_gpu
    if hasattr(globals()[var], 'free_gpu'):
      try: globals()[var].free_gpu()
      except:pass

    del globals()[var]
# 2. Force Python garbage collection
gc.collect()

# 3. Clear the PyTorch cache
if torch.cuda.is_available():
  torch.cuda.empty_cache()
  torch.cuda.reset_peak_memory_stats()
print(f"GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MiB allocated")
"""

In [46]:
try:
    import optuna
except ImportError as _e:
    raise ImportError('pip install optuna') from _e

for _v in ['train_wallet_ae','score_wallet_ae','make_wallet_loader','TR_CFG','IF_CFG',
            'WALLET_INDEX','X_tx','X_tx_norm','tx_feature_cols','MAX_SEQ_LEN',
            'train_wallets','val_wallets','train_mask_tx']:
    if _v not in globals(): raise RuntimeError(f'Missing prerequisite: {_v}')

optuna.logging.set_verbosity(optuna.logging.WARNING)
OPTUNA_N_TRIALS, OPTUNA_TIMEOUT, OPTUNA_APPLY_BEST = 30, 3600, True

_base_if_cfg = {**IF_CFG, 'contamination': IF_CFG.get('contamination', 0.005), 'random_state': 42}
_if_train_n  = int(train_mask_tx.sum())
assert _if_train_n > 0 and len(val_wallets) > 0, 'Empty train/val split'


# 2. Updated Optuna Objective (passing the trial object)
def _optuna_objective(trial):
    d_model      = int(trial.suggest_categorical('ae_d_model', [32, 48, 64, 96, 128]))
    nhead        = int(trial.suggest_categorical('ae_nhead', [h for h in [2,4,8] if d_model % h == 0]))
    num_layers   = int(trial.suggest_int('ae_num_layers', 1, 4))
    ff_mult      = int(trial.suggest_int('ae_ff_mult', 2, 4))
    dropout      = float(trial.suggest_float('ae_dropout', 0.05, 0.40))
    lr           = float(trial.suggest_float('ae_lr', 5e-5, 5e-3, log=True))
    weight_decay = float(trial.suggest_float('ae_weight_decay', 1e-8, 1e-2, log=True))
    if_n_est     = int(trial.suggest_int('if_n_estimators', 100, 800, step=50))
    if_max_samp  = min(int(trial.suggest_categorical('if_max_samples', [128,256,512,1024,2048])), _if_train_n)

    tr_cfg = {
        **TR_CFG,
        'd_model': d_model,
        'nhead': nhead,
        'num_layers': num_layers,
        'ff_dim': d_model * ff_mult,
        'dropout': dropout,
        'lr': lr,
        'weight_decay': weight_decay,
        'epochs': 30,
        'batch_size': min(TR_CFG.get('batch_size', 512), 512)
    }

    _rng    = np.random.default_rng(trial.number + 99)
    _sub_w  = _rng.choice(train_wallets, size=len(train_wallets)//2, replace=False).tolist()
    _sub_tx = df_flat['wallet'].is_in(_sub_w).to_numpy() & train_mask_tx

    # IF: fit on subsampled train txs, CDF-normalise against same subset
    if_model = IsolationForest(
        n_estimators=if_n_est,
        max_samples=if_max_samp,
        contamination=_base_if_cfg['contamination'],
        random_state=42
    ).fit(X_tx[_sub_tx])

    if_raw   = -if_model.decision_function(X_tx)
    if_tx_s  = np.searchsorted(np.sort(if_raw[_sub_tx]), if_raw, side='right') / _sub_tx.sum()

    # Aggregate IF tx scores → wallet-max for val wallets
    _val_w_list = val_wallets.tolist()
    _w2max = (
        pl.DataFrame({'wallet': df_flat['wallet'], 'score': if_tx_s})
        .filter(pl.col('wallet').is_in(_val_w_list))
        .group_by('wallet').agg(pl.col('score').max())
        .to_pandas().set_index('wallet')['score'].to_dict()
    )

    if_wal_val = np.array([_w2max.get(w, 0.0) for w in _val_w_list], dtype=np.float32)

    # AE: train on subsampled wallets, score val wallets
    _tr_loader = make_wallet_loader(
        X_tx_norm, WALLET_INDEX, _sub_w, tx_feature_cols,
        MAX_SEQ_LEN, batch_size=tr_cfg['batch_size']
    )

    _val_loader = make_wallet_loader(
        X_tx_norm, WALLET_INDEX, _val_w_list, tx_feature_cols,
        MAX_SEQ_LEN, batch_size=tr_cfg['batch_size'], shuffle=False
    )

    _m, _trl, _vll = train_wallet_ae(
        _tr_loader, _val_loader, tr_cfg,
        n_features=len(tx_feature_cols),
        max_seq_len=MAX_SEQ_LEN,
        device=device,
        trial=trial
    )

    tr_wal_val, _ = score_wallet_ae(
        _m, X_tx_norm, WALLET_INDEX, _val_w_list,
        MAX_SEQ_LEN, device, tx_feature_cols
    )

    val_mse   = float(_vll[-1]) if _vll else 1e6
    gen_gap   = max(0.0, val_mse - (float(_trl[-1]) if _trl else 1e6))
    ae_spread = float(
        np.log1p(np.quantile(tr_wal_val, 0.99)) -
        np.log1p(np.median(tr_wal_val) + 1e-8)
    )

    _if_top = set(np.where(if_wal_val >= np.quantile(if_wal_val, 0.99))[0])
    _tr_top = set(np.where(tr_wal_val >= np.quantile(tr_wal_val, 0.99))[0])
    diversity = 1.0 - len(_if_top & _tr_top) / max(len(_if_top | _tr_top), 1)

    obj = (
        0.55 * ae_spread +
        0.30 * diversity -
        0.10 * np.log1p(val_mse * 100) -
        0.05 * np.log1p(gen_gap * 100)
    )

    for k, v in [
        ('ae_spread', ae_spread),
        ('diversity', diversity),
        ('val_mse', val_mse),
        ('gen_gap', gen_gap)
    ]:
        trial.set_user_attr(k, float(v))

    del _m
    getattr(if_model, "free_gpu", lambda: None)()
    del if_model
    torch.cuda.empty_cache()
    import gc; gc.collect()

    return float(obj)


# 3. Updated Study Creation (Hyperband Pruner)
_study_db = 'sqlite:////content/drive/MyDrive/Colab-Notebooks/vastaiFinal/optuna_study.db'
os.makedirs('/content/drive/MyDrive/Colab-Notebooks/vastaiFinal', exist_ok=True)

pruner = optuna.pruners.HyperbandPruner(
    min_resource=3,
    max_resource=30,
    reduction_factor=3
)

optuna_study = optuna.create_study(
    study_name='vastaifloans',
    storage=_study_db,
    load_if_exists=True,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
    pruner=pruner
)

_done = len(optuna_study.trials)
print(f'Optuna: {OPTUNA_N_TRIALS} trials (done={_done}, remaining={OPTUNA_N_TRIALS-_done})')

if OPTUNA_N_TRIALS - _done > 0:
    optuna_study.optimize(
        _optuna_objective,
        n_trials=OPTUNA_N_TRIALS - _done,
        timeout=OPTUNA_TIMEOUT,
        gc_after_trial=True
    )
else:
    print('All trials complete.')

_best = optuna_study.best_trial
print(f'\nBest objective : {_best.value:.6f}\nBest params    : {_best.params}')

optuna_best_if_cfg = {k.replace('if_',''): v for k,v in _best.params.items() if k.startswith('if_')}
optuna_best_if_cfg.update({'contamination': _base_if_cfg['contamination'], 'random_state': 42})
optuna_best_tr_cfg = {k.replace('ae_',''): v for k,v in _best.params.items() if k.startswith('ae_')}
optuna_best_tr_cfg['ff_dim'] = int(optuna_best_tr_cfg['d_model'] * optuna_best_tr_cfg.pop('ff_mult'))
optuna_best_tr_cfg.update({'epochs': 30, 'batch_size': TR_CFG['batch_size'],
                            'mask_ratio': TR_CFG.get('mask_ratio', 0.15)})
TARGET_ALPHA = 0.01

if OPTUNA_APPLY_BEST:
    print('\nApplying best params and retraining...')
    IF_CFG      = dict(optuna_best_if_cfg)
    IF_CFG['n_jobs'] = 4
    iforest     = IsolationForest(**IF_CFG).fit(X_tx[train_mask_tx])
    if_raw      = -iforest.decision_function(X_tx)
    if_tx_score = np.searchsorted(np.sort(if_raw[train_mask_tx]), if_raw, side='right') / train_mask_tx.sum()

    TR_CFG  = dict(optuna_best_tr_cfg)
    _all_w  = df_flat['wallet'].unique().to_list()
    train_loader = make_wallet_loader(X_tx_norm, WALLET_INDEX, train_wallets.tolist(), tx_feature_cols,
                                      MAX_SEQ_LEN, batch_size=TR_CFG['batch_size'])
    val_loader   = make_wallet_loader(X_tx_norm, WALLET_INDEX, val_wallets.tolist(), tx_feature_cols,
                                      MAX_SEQ_LEN, batch_size=TR_CFG['batch_size'], shuffle=False)
    tr_model, ae_train_losses, ae_val_losses = train_wallet_ae(
        train_loader, val_loader, TR_CFG,
        n_features=len(tx_feature_cols), max_seq_len=MAX_SEQ_LEN, device=device)
    tr_wallet_raw, per_feat_errors = score_wallet_ae(
        tr_model, X_tx_norm, WALLET_INDEX, _all_w, MAX_SEQ_LEN, device, tx_feature_cols)
    _train_w_mask = np.isin(np.array(_all_w), np.array(train_wallets))
    tr_tx_score   = np.searchsorted(np.sort(tr_wallet_raw[_train_w_mask]),
                                    tr_wallet_raw, side='right') / _train_w_mask.sum()
    print('IF_CFG, TR_CFG, if_tx_score, tr_tx_score updated.')
else:
    print('OPTUNA_APPLY_BEST=False — results in optuna_best_if_cfg / optuna_best_tr_cfg.')

Optuna: 30 trials (done=0, remaining=30)
  Epoch 1/30  train=1.510516  val=0.828445
  Epoch 2/30  train=0.980082  val=0.772169
  Epoch 3/30  train=0.980550  val=0.779973
  Epoch 4/30  train=0.975305  val=0.791924
  Epoch 5/30  train=0.943909  val=0.783470
  Early stop at epoch 5. Restoring best weights.
  Epoch 1/30  train=2.259530  val=0.876635
  Epoch 2/30  train=0.999159  val=0.797042
  Epoch 3/30  train=0.949889  val=0.753657
  Epoch 4/30  train=0.917013  val=0.751139
  Epoch 1/30  train=5.508328  val=4.335286
  Epoch 2/30  train=4.164152  val=3.312417
  Epoch 3/30  train=3.196987  val=2.472809
  Epoch 4/30  train=2.439601  val=1.817585
  Epoch 5/30  train=1.887394  val=1.393746
  Epoch 6/30  train=1.557324  val=1.147886
  Epoch 7/30  train=1.378936  val=1.023587
  Epoch 8/30  train=1.249055  val=0.952957
  Epoch 9/30  train=1.175243  val=0.910213
  Epoch 10/30  train=1.172398  val=0.897965
  Epoch 11/30  train=1.124743  val=0.866403
  Epoch 12/30  train=1.112475  val=0.871525
  Ep

In [59]:
# ── Save models ───────────────────────────────────────────────────────────────
import os, joblib, pickle

CKPT = '/content/drive/MyDrive/Colab-Notebooks/vastaiFinal/checkpoints/'
os.makedirs(CKPT, exist_ok=True)

# Optuna study (best params + all trial results)
with open(CKPT + 'optuna_study.pkl', 'wb') as _f:
  pickle.dump(optuna_study, _f)
print('Saved: optuna_study.pkl')

# Isolation Forest
# WORKAROUND: Drop the unpicklable GPU/C++ pointer before saving.
# This preserves the trained scikit-learn trees underneath.
if hasattr(iforest, '_fil'):
  iforest._fil = None

joblib.dump(iforest, CKPT + 'iforest.pkl')
print('Saved: iforest.pkl')

# Scaler
joblib.dump(scaler, CKPT + 'scaler.pkl')
print('Saved: scaler.pkl')

# Transformer AE weights
_ae_state = tr_model.module.state_dict() if hasattr(tr_model, 'module') else tr_model.state_dict()
torch.save({'state_dict': _ae_state, 'cfg': TR_CFG}, CKPT + 'ae_model.pt')
print('Saved: ae_model.pt')

Saved: optuna_study.pkl


PicklingError: Can't pickle <class '__main__.IsolationForest'>: it's not the same object as __main__.IsolationForest

In [50]:
# ── Checkpoint: save Optuna study + best models + scores to Drive ────────────
import os, joblib, torch

CKPT = '/content/drive/MyDrive/Colab-Notebooks/vastaiFinal/checkpoints'
os.makedirs(CKPT, exist_ok=True)

# Optuna study (best params + all trial results)
import pickle
with open(CKPT + 'optuna_study.pkl', 'wb') as _f:
    pickle.dump(optuna_study, _f)
print('Saved: optuna_study.pkl')

# Isolation Forest
joblib.dump(iforest, CKPT + 'iforest.pkl')
print('Saved: iforest.pkl')

# Scaler
joblib.dump(scaler, CKPT + 'scaler.pkl')
print('Saved: scaler.pkl')

# Transformer AE weights
_ae_state = tr_model.module.state_dict() if hasattr(tr_model, 'module') else tr_model.state_dict()
torch.save({'state_dict': _ae_state, 'cfg': TR_CFG}, CKPT + 'ae_model.pt')
print('Saved: ae_model.pt')

# Scores
import numpy as np
np.save(CKPT + 'if_tx_score.npy', if_tx_score)
np.save(CKPT + 'tr_tx_score.npy', tr_tx_score)
np.save(CKPT + 'per_feat_errors.npy', per_feat_errors)
print('Saved: if_tx_score.npy, tr_tx_score.npy, per_feat_errors.npy')

print(f'\nAll checkpoints saved to {CKPT}')


Saved: optuna_study.pkl


PicklingError: Can't pickle <class '__main__.IsolationForest'>: it's not the same object as __main__.IsolationForest

In [60]:
# ── Combine scores + aggregate to wallet level ────────────────────────────────
#
# Fusion formula (per transaction t, wallet w):
#
#   combined(t) = W_IF · if_score(t)  +  W_TR · tr_score(t)
#
# where:
#   if_score(t)  = rank-normalised Isolation Forest anomaly score ∈ (0, 1]
#   tr_score(t)  = min-max-normalised Transformer AE reconstruction MSE ∈ [0, 1]
#   W_IF = W_TR  = 0.5
#
# Wallet-level aggregation (max over transactions):
#   raw_score(w)  = max_t  combined(t)
#   if_score(w)   = max_t  if_score(t)
#   tr_score(w)   = max_t  tr_score(t)
#   p90_score(w)  = p90_t  combined(t)

W_IF, W_TR = 0.5, 0.5

# FIX: tr_tx_score is at the wallet level (N=619,160).
# We broadcast it back to the transaction level (N=2,905,990) by mapping the wallet IDs.
wallet_to_tr_score = dict(zip(_all_wallets, tr_tx_score))
tr_tx_score_mapped = np.array([wallet_to_tr_score[w] for w in df_flat['wallet']], dtype=np.float32)

# Now both arrays are shape (2905990,) and can be safely combined
combined_tx = np.maximum(if_tx_score, tr_tx_score_mapped)

# Assign back to df_flat (Handles both Pandas and Polars states securely)
if hasattr(df_flat, "with_columns"):
  df_flat = df_flat.with_columns([
      pl.Series('if_tx_score', if_tx_score),
      pl.Series('tr_tx_score', tr_tx_score_mapped),
      pl.Series('combined_tx_score', combined_tx)
  ])
  _for_agg = df_flat.select(['wallet', 'if_tx_score', 'tr_tx_score', 'combined_tx_score']).to_pandas()
else:
  df_flat['if_tx_score']       = if_tx_score
  df_flat['tr_tx_score']       = tr_tx_score_mapped
  df_flat['combined_tx_score'] = combined_tx
  _for_agg = df_flat[['wallet', 'if_tx_score', 'tr_tx_score', 'combined_tx_score']]

wallet_scores = (pl.from_pandas(_for_agg)
  .group_by("wallet")
  .agg([
      pl.col("if_tx_score").max().alias("if_score"),
      pl.col("tr_tx_score").max().alias("tr_score"),
      pl.col("combined_tx_score").max().alias("raw_score"),
      pl.col("combined_tx_score").quantile(0.9).alias("p90_score"),
      pl.col("combined_tx_score").count().alias("n_tx")
  ])).to_pandas()

# Explicit linear blend of per-wallet maxes — highlights when argmax tx differs
# between IF and AE (raw_score > linear_blend means they peak on different txs).
wallet_scores['linear_blend'] = (W_IF * wallet_scores['if_score']
                                + W_TR * wallet_scores['tr_score'])
wallet_scores['argmax_divergence'] = (wallet_scores['raw_score']
                                    - wallet_scores['linear_blend']).round(4)

print(f'Wallet scores: {len(wallet_scores):,}')
print(wallet_scores[['if_score','tr_score','raw_score','linear_blend','argmax_divergence']]
    .describe(percentiles=[.5,.9,.99]).round(4))
n_div = (wallet_scores['argmax_divergence'] > 0.01).sum()
print(f'\nWallets where IF/AE peak on different transactions: {n_div:,} '
    f'({n_div/len(wallet_scores)*100:.1f}%)')

Wallet scores: 619,160
          if_score     tr_score    raw_score  linear_blend  argmax_divergence
count  619160.0000  619160.0000  619160.0000   619160.0000        619160.0000
mean        0.4406       0.5036       0.6244        0.4721             0.1523
std         0.1563       0.3152       0.2218        0.1758             0.0937
min         0.0817       0.0000       0.1152        0.0985             0.0000
50%         0.4027       0.5201       0.5988        0.4789             0.1435
90%         0.6459       0.9361       0.9498        0.6872             0.2822
99%         0.9876       0.9898       0.9970        0.8921             0.4006
max         1.0000       1.0000       1.0000        0.9992             0.5000

Wallets where IF/AE peak on different transactions: 599,643 (96.8%)


In [ ]:
# ── Checkpoint: save combined scores + wallet scores ─────────────────────────
import os, numpy as np

CKPT = '/workspace/checkpoints/'
os.makedirs(CKPT, exist_ok=True)

np.save(CKPT + 'combined_tx_score.npy', combined_tx)
print('Saved: combined_tx_score.npy')

wallet_scores.to_parquet(CKPT + 'wallet_scores.parquet', index=False)
print('Saved: wallet_scores.parquet')

In [ ]:
# ── Alpha selection (pre-conformal) ───────────────────────────────────────────
# Select ALPHA before conformal calibration using wallet score distribution.
# Full synthetic detection validation follows in the contamination cell.
#
# Method: kneedle algorithm on log(alpha) vs flag_rate.
# Alpha is a log-scale parameter so x-axis is log(alpha).
# For a concave curve the knee = max(x_norm - y_norm) — the point furthest
# below the diagonal, where raising alpha stops buying proportional flag-rate
# gain. Validated against synthetic detection rates in the contamination cell.

_val_mask_w  = wallet_scores['wallet'].isin(val_wallets)
_test_mask_w = wallet_scores['wallet'].isin(test_wallets)
_cal_pre  = wallet_scores.loc[_val_mask_w,  'raw_score'].values
_test_pre = wallet_scores.loc[_test_mask_w, 'raw_score'].values

_alpha_candidates = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
_alpha_rows = []
for _a in _alpha_candidates:
  _thr  = float(np.quantile(_cal_pre, 1 - _a))
  _rate = float((_test_pre >= _thr).mean() * 100)
  _alpha_rows.append({'alpha': _a, 'threshold': round(_thr, 6), 'flag_rate_pct': round(_rate, 2)})

_alpha_df = pd.DataFrame(_alpha_rows)
print('Alpha vs flag rate (wallet level):')
print(_alpha_df.to_string(index=False))

# Kneedle on log(alpha) vs flag_rate
_log_a = np.log(_alpha_df['alpha'].values)
_flag  = _alpha_df['flag_rate_pct'].values
_x = (_log_a - _log_a[0]) / (_log_a[-1] - _log_a[0])
_y = (_flag  - _flag[0])  / (_flag[-1]  - _flag[0])
_knee_idx = int(np.argmax(_x - _y))
ALPHA = float(_alpha_df.iloc[_knee_idx]['alpha'])
print(f'\nSelected ALPHA={ALPHA} '
    f'(kneedle on log-alpha vs flag-rate, '
    f'flag_rate={_alpha_df.iloc[_knee_idx]["flag_rate_pct"]:.2f}%)')


In [ ]:
# ── Data-driven alpha selection ───────────────────────────────────────────────
# Dataset is pre-filtered cascade transactions (multi-protocol DeFi hops).
# Base rate is NOT general Solana tx (~0.15-2%) but attack rate within
# cascade-filtered population — likely 3-10%. MAX_FLAG_RATE=0.10 reflects this.
#
# IMPORTANT: GMM must be fit on RAW scores (if_raw, tr_raw), NOT rank-normalized
# scores. Rank-normalized scores are uniform by construction — a GMM on them
# will never find natural separation. Raw IF decision function and raw AE MSE
# have heavy tails that allow Gaussians to actually separate outliers from normal.
#
# Priority:
#   1. GMM on raw IF scores  — natural split in IF decision function space
#   2. GMM on raw AE MSE     — natural split in reconstruction error space
#   3. IF contamination prior — fallback if both GMMs find no separation

from sklearn.mixture import GaussianMixture

MAX_FLAG_RATE = 0.10  # cascade-filtered dataset: up to 10% exploits plausible

# ── Aggregate raw scores to wallet level (max over transactions) ──────────────
_wdf = df_flat[['wallet']].copy()
_wdf['if_raw_score'] = if_raw
_wdf['tr_raw_score'] = tr_raw

_wallet_raw = _wdf.groupby('wallet').agg(
  if_raw_max=('if_raw_score', 'max'),
  tr_raw_max=('tr_raw_score', 'max'),
)

_val_if_raw = _wallet_raw.loc[_wallet_raw.index.isin(val_wallets), 'if_raw_max'].values
_val_tr_raw = _wallet_raw.loc[_wallet_raw.index.isin(val_wallets), 'tr_raw_max'].values

def _fit_gmm_alpha(raw_scores, label):
  """Fit 2-component GMM on raw scores, return (alpha, gap, boundary)."""
  _sc  = raw_scores.reshape(-1, 1)
  _g   = GaussianMixture(n_components=2, random_state=42).fit(_sc)
  _ord = np.argsort(_g.means_.flatten())
  _mn, _mx = _g.means_.flatten()[_ord]
  _wn, _wx = _g.weights_.flatten()[_ord]

  # Decision boundary: P(anomalous | x) = 0.5
  _grid = np.linspace(raw_scores.min(), raw_scores.max(), 10000)
  _resp = _g.predict_proba(_grid.reshape(-1, 1))
  _ci   = np.argmin(np.abs(_resp[:, _ord[1]] - 0.5))
  _thr  = float(_grid[_ci])

  _above = raw_scores[raw_scores >= _thr]
  _below = raw_scores[raw_scores <  _thr]
  _gap   = float(_above.min() - _below.max()) if len(_above) and len(_below) else 0.0
  _alpha = float(np.mean(raw_scores >= _thr))

  print(f'{label} normal  : mean={_mn:.4f}  weight={_wn:.3f}')
  print(f'{label} anomaly : mean={_mx:.4f}  weight={_wx:.3f}')
  print(f'{label} boundary: {_thr:.4f}  gap={_gap:.5f}  alpha={_alpha:.4f}')
  return _alpha, _gap, _thr, _wx

print('── GMM on raw IF decision function (val wallets) ──')
_if_alpha, _if_gap, _if_thr, _if_w = _fit_gmm_alpha(_val_if_raw, 'IF')

print('\n── GMM on raw AE MSE (val wallets) ──')
_tr_alpha, _tr_gap, _tr_thr, _tr_w = _fit_gmm_alpha(_val_tr_raw, 'AE')

# ── IF contamination prior ────────────────────────────────────────────────────
_if_cont = IF_CFG.get('contamination', 0.005)
_if_cont = 0.005 if _if_cont == 'auto' else float(_if_cont)
print(f'\nIF contamination prior: {_if_cont:.4f}')

# ── Candidate grid ────────────────────────────────────────────────────────────
# Dense sweep anchored around GMM estimates + IF prior, clipped to budget
_gmm_anchor = np.mean([_if_alpha, _tr_alpha])
_candidates = sorted(set(
  # Fine sweep around GMM anchors (±50% in 0.5% steps)
  [round(a, 3) for a in np.arange(
      max(0.001, _gmm_anchor * 0.5),
      min(MAX_FLAG_RATE, _gmm_anchor * 1.5) + 0.001,
      0.005
  )]
  # Add GMM boundary estimates directly
  + [round(_if_alpha, 3), round(_tr_alpha, 3),
     round((_if_alpha + _tr_alpha) / 2, 3)]
  # Add IF prior + standard references
  + [round(_if_cont, 3), 0.005, 0.01, 0.02, 0.05]
))
_candidates = [a for a in _candidates if 0 < a <= MAX_FLAG_RATE]

_val_sc = wallet_scores.loc[wallet_scores['wallet'].isin(val_wallets), 'raw_score'].values
_alpha_rows = []
for _a in _candidates:
  _thr  = np.quantile(_val_sc, 1 - _a)
  _ab   = _val_sc[_val_sc >= _thr]
  _bl   = _val_sc[_val_sc <  _thr]
  _gap  = float(_ab.min() - _bl.max()) if len(_ab) and len(_bl) else 0.0
  _src  = ('gmm_if'    if _a == round(_if_alpha, 3)
           else 'gmm_ae'   if _a == round(_tr_alpha, 3)
           else 'blend'    if _a == round((_if_alpha + _tr_alpha) / 2, 3)
           else 'if_prior' if _a == round(_if_cont, 3)
           else 'reference')
  _alpha_rows.append({
      'alpha': _a, 'threshold': round(_thr, 4),
      'n_flagged_val': int((_val_sc >= _thr).sum()),
      'score_gap': round(_gap, 5), 'source': _src,
  })

alpha_df = pd.DataFrame(_alpha_rows)
print('\nAlpha candidate grid (on rank-normalized combined scores, for reference):')
print(alpha_df.to_string(index=False))

# ── Selection priority ────────────────────────────────────────────────────────
if _if_gap > 0 and round(_if_alpha, 3) <= MAX_FLAG_RATE:
  ALPHA   = round(_if_alpha, 3)
  _reason = f'GMM on raw IF scores (gap={_if_gap:.5f})'
elif _tr_gap > 0 and round(_tr_alpha, 3) <= MAX_FLAG_RATE:
  ALPHA   = round(_tr_alpha, 3)
  _reason = f'GMM on raw AE MSE (gap={_tr_gap:.5f})'
elif round((_if_alpha + _tr_alpha) / 2, 3) <= MAX_FLAG_RATE:
  ALPHA   = round((_if_alpha + _tr_alpha) / 2, 3)
  _reason = 'IF/AE GMM blend (no clean gap in either)'
else:
  ALPHA   = _if_cont
  _reason = 'IF contamination prior (fallback — GMMs found no separation)'

if _if_w > 0.15 and _tr_w > 0.15:
  print('\nWARN: Both GMM anomaly weights > 15% — model may not be separating well.')
  print('Consider re-tuning IF contamination or Optuna objective.')

print(f'\nData-driven alpha : {ALPHA}  [{_reason}]')
print(f'Hard budget cap   : {MAX_FLAG_RATE} (cascade-filtered; override if needed)')
print('(Override ALPHA manually if domain knowledge suggests otherwise)')

In [ ]:
# ── Conformal prediction (wallet level) ───────────────────────────────────────
# Calibrate on val wallets; compute p-values for test wallets.
# Theoretical guarantee: FPR ≤ alpha under exchangeability assumption.

ALPHA = 0.02

val_mask_w  = wallet_scores['wallet'].isin(val_wallets)
test_mask_w = wallet_scores['wallet'].isin(test_wallets)

cal_scores  = wallet_scores.loc[val_mask_w,  'raw_score'].values
test_scores = wallet_scores.loc[test_mask_w, 'raw_score'].values

conformal_threshold = np.quantile(cal_scores, 1 - ALPHA)
test_pvals = np.array([
    (np.sum(cal_scores >= s) + 1) / (len(cal_scores) + 1)
    for s in test_scores
])

n_flagged = (test_pvals <= ALPHA).sum()
print(f'Conformal threshold (alpha={ALPHA}): {conformal_threshold:.4f}')
print(f'Test wallets flagged : {n_flagged} / {len(test_scores)} '
      f'({n_flagged/len(test_scores)*100:.2f}%)')

wallet_scores.loc[test_mask_w, 'conformal_p']    = test_pvals
wallet_scores.loc[test_mask_w, 'conformal_flag'] = (test_pvals <= ALPHA).astype(int)

In [ ]:
# ── Conformal validity: empirical coverage & temporal exchangeability ─────────
# Under valid exchangeability, LOO p-values on cal set should be uniform → flag
# rate ≈ alpha. A KS test on score distributions over time checks temporal drift.

from scipy.stats import ks_2samp

# ── Recalibrate on most-recent val wallets (time-stratified) ─────────────────
# Standard random split violates exchangeability when scores drift over time.
# Fix: use the latest 50% of val wallets (by median block_timestamp) as cal set
# so the threshold is anchored to recent distribution.
_val_wallet_times = (
  df_flat[df_flat['wallet'].isin(val_wallets)]
  .groupby('wallet')['block_timestamp'].median()
  .sort_values()
)
_cal_cutoff = _val_wallet_times.quantile(0.50)
cal_wallets_ts  = set(_val_wallet_times[_val_wallet_times >= _cal_cutoff].index)
early_val_wallets = set(_val_wallet_times[_val_wallet_times < _cal_cutoff].index)

cal_mask   = wallet_scores['wallet'].isin(cal_wallets_ts)
cal_scores = wallet_scores.loc[cal_mask, 'raw_score'].values
print(f'Time-stratified cal set: {len(cal_scores):,} wallets (recent 50% of val)')

# Recompute conformal threshold on time-stratified cal set
conformal_threshold = float(np.quantile(cal_scores, 1 - ALPHA))
print(f'Updated conformal threshold: {conformal_threshold:.4f}')

# 1) LOO conformal p-values — correct vectorised formula
# p_i = |{j in cal : s_j >= s_i}| / n  (includes self → valid LOO approximation)
print(f'Computing LOO p-values on {len(cal_scores):,} cal wallets...')
_cal_sorted = np.sort(cal_scores)
_cal_loo_p  = (len(cal_scores) - np.searchsorted(_cal_sorted, cal_scores, side='left')) / len(cal_scores)

_realized_fpr = float((_cal_loo_p <= ALPHA).mean())
print(f'Target alpha              : {ALPHA}')
print(f'Realised flag rate on cal : {_realized_fpr:.4f}  '
    f'({("OK" if abs(_realized_fpr - ALPHA) < 0.01 else "WARN — deviates from alpha")})')

# 2) Temporal exchangeability: KS test across three equal time-thirds of val set
_val_df     = df_flat[df_flat['wallet'].isin(val_wallets)].sort_values('block_timestamp')
_thirds_idx = np.array_split(_val_df.index, 3)
_thirds_sc  = [_val_df.loc[_idx, 'combined_tx_score'].values for _idx in _thirds_idx]
_ks_stat, _ks_p = ks_2samp(_thirds_sc[0], _thirds_sc[2])
print(f'\nKS test (val early vs late third): stat={_ks_stat:.4f}  p={_ks_p:.4e}')
if _ks_p < 0.05:
  print('  WARN: score distribution drifts over time — using time-stratified cal mitigates this.')
  print('  For stronger guarantees consider Adaptive Conformal Prediction (Gibbs & Candès, 2021).')
else:
  print('  OK: no significant temporal drift detected.')

# Seaborn plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# p-value histogram (should be uniform)
sns.histplot(_cal_loo_p, bins=20, color='#4C78A8', edgecolor='white', ax=axes[0])
axes[0].axhline(len(_cal_loo_p) / 20, color='red', linestyle='--', label='Expected (uniform)')
axes[0].set_xlabel('LOO Conformal p-value')
axes[0].set_ylabel('Count')
axes[0].set_title('Cal-Set p-value Distribution\n(uniform = valid coverage)')
axes[0].legend()

# CDF by time-third
_cdf_parts = []
for _sc, _lbl in zip(_thirds_sc, ['early', 'mid', 'late']):
  _ss = np.sort(_sc)
  if len(_ss) == 0:
      continue
  # Downsample to 1000 points for fast rendering
  _idx = np.linspace(0, len(_ss) - 1, min(1000, len(_ss))).astype(int)
  _cdf_parts.append(pd.DataFrame({
      'score': _ss[_idx],
      'cdf': np.linspace(0, 1, len(_idx)),
      'period': _lbl
  }))

if len(_cdf_parts):
  _cdf_df = pd.concat(_cdf_parts, ignore_index=True)
  sns.lineplot(data=_cdf_df, x='score', y='cdf', hue='period', ax=axes[1])
axes[1].axvline(conformal_threshold, color='black', linestyle='--', linewidth=1, label=f'threshold (a={ALPHA})')
axes[1].set_xlabel('Combined score')
axes[1].set_ylabel('CDF')
axes[1].set_title('Score CDF by Time Period (val)\nDivergence -> exchangeability violation')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('conformal_validity.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: conformal_validity.png')

# ── Temporal Decay & Adaptive Conformal Update ────────────────────────────────
# To handle concept drift in non-stationary blockchain environments, we implement a rolling window analysis.
# A rigid alpha fails under drift; an adaptive alpha adjusts dynamically based on the error rate in the previous window.

In [ ]:
# ── 5. Rolling Window Split Logic ──────────────────────────────────────────
# We dynamically define rolling windows across the dataset's available months.
# Each window consists of (Train Month, Val Month, Test Month).

# Derive _month from block_timestamp if not already present
if '_month' not in df_flat.columns:
  df_flat['_month'] = df_flat['block_timestamp'].dt.to_period('M').astype(str)

all_months = sorted(df_flat['_month'].unique().astype(str).tolist())
print(f'Available months in dataset: {all_months}')

if len(all_months) < 3:
  raise ValueError(f"Need at least 3 months for a rolling window, found {len(all_months)}: {all_months}")

rolling_windows = []
for i in range(len(all_months) - 2):
  rolling_windows.append((all_months[i], all_months[i+1], all_months[i+2]))

print(f'Defined {len(rolling_windows)} rolling windows:')
for i, w in enumerate(rolling_windows):
  print(f'  W{i+1}: Train={w[0]}, Val={w[1]}, Test={w[2]}')

def get_window_masks(df, train_m, val_m, test_m):
  train_w = df[df["_month"] == train_m]["wallet"].unique()
  val_w   = df[df["_month"] == val_m]["wallet"].unique()
  test_w  = df[df["_month"] == test_m]["wallet"].unique()

  # Ensure no overlap for strict evaluation
  val_w  = np.setdiff1d(val_w, train_w)
  test_w = np.setdiff1d(test_w, np.union1d(train_w, val_w))

  m_train = df["wallet"].isin(train_w) & (df["_month"] == train_m)
  m_val   = df["wallet"].isin(val_w)   & (df["_month"] == val_m)
  m_test  = df["wallet"].isin(test_w)  & (df["_month"] == test_m)

  return m_train, m_val, m_test, train_w, val_w, test_w

# Initialize masks and wallet lists for the first window for baseline setup
train_m, val_m, test_m = rolling_windows[0]
train_mask_tx, val_mask_tx, test_mask_tx, train_wallets, val_wallets, test_wallets = get_window_masks(df_flat, train_m, val_m, test_m)

In [ ]:
#── 6. Model Decay & Adaptive Conformal Analysis ─────────────────────────────
# Evaluates how the fixed model architecture decays over time and
# how Adaptive Conformal Inference (ACI) stabilizes the detection rate.

import gc

decay_results = []
GAMMA = 0.05  # ACI learning rate

for i, (m1, m2, m3) in enumerate(rolling_windows):
  print(f'\nProcessing Window {i+1}: Train={m1}, Val={m2}, Test={m3}')

  # Get masks for this window
  m_train, m_val, m_test, tw, vw, tst_w = get_window_masks(df_flat, m1, m2, m3)

  # Re-train models using best Optuna params (fixed architecture)
  if_win = IsolationForest(**optuna_best_if_cfg).fit(X_tx[m_train])
  if_raw_win = -if_win.decision_function(X_tx)
  _if_sort = np.sort(if_raw_win[m_train])
  if_score_win = np.searchsorted(_if_sort, if_raw_win, side='right') / len(_if_sort)

  tr_model_win, _, _ = train_tx_ae(X_tx_norm[m_train], X_tx_norm[m_val], optuna_best_tr_cfg, device)
  tr_raw_win, _ = score_tx_ae(tr_model_win, X_tx_norm, device)
  _tr_sort = np.sort(tr_raw_win[m_train])
  tr_score_win = np.searchsorted(_tr_sort, tr_raw_win, side='right') / len(_tr_sort)

  # Combined score
  comb_win = np.maximum(if_score_win, tr_score_win)

  # A. Rigid threshold: set on train, applied to test unchanged
  train_scores     = comb_win[m_train]
  threshold_rigid  = np.quantile(train_scores, 1 - TARGET_ALPHA)

  # B. Check concept drift on val
  val_scores         = comb_win[m_val]
  empirical_rate_val = float(np.mean(val_scores >= threshold_rigid))  # rate, not count

  # C. Adaptive Conformal (ACI): adjust alpha based on val drift
  alpha_adapt     = TARGET_ALPHA + GAMMA * (TARGET_ALPHA - empirical_rate_val)
  alpha_adapt     = float(np.clip(alpha_adapt, 0.001, 0.2))
  threshold_adapt = np.quantile(val_scores, 1 - alpha_adapt)

  # D. Evaluate on test
  test_scores  = comb_win[m_test]
  flags_rigid  = int(np.sum(test_scores >= threshold_rigid))
  flags_adapt  = int(np.sum(test_scores >= threshold_adapt))

  decay_results.append({
      'window':          f'W{i+1}',
      'train_month':     m1,
      'val_month':       m2,
      'test_month':      m3,
      'val_error_rate':  round(empirical_rate_val, 4),
      'adaptive_alpha':  round(alpha_adapt, 4),
      'rigid_flags':     flags_rigid,
      'adapt_flags':     flags_adapt,
      'flag_rate_rigid': round(flags_rigid / max(len(test_scores), 1), 4),
      'flag_rate_adapt': round(flags_adapt / max(len(test_scores), 1), 4),
  })

  # Free GPU memory before next window
  del tr_model_win
  if_win.free_gpu()
  del if_win
  torch.cuda.empty_cache()
  gc.collect()

decay_df = pd.DataFrame(decay_results)
print('\n=== Model Decay Summary ===')
print(decay_df[['window', 'train_month', 'test_month', 'val_error_rate',
              'adaptive_alpha', 'flag_rate_rigid', 'flag_rate_adapt']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(12, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

windows = decay_df['window'].tolist()
x = range(len(windows))
labels = [f"{r['train_month']}\n→{r['test_month']}" for _, r in decay_df.iterrows()]

# 1. Flag rate: rigid vs adaptive
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(x, decay_df['flag_rate_rigid'] * 100, 'o--', color='tomato',   label='Rigid threshold')
ax1.plot(x, decay_df['flag_rate_adapt'] * 100, 's-',  color='steelblue', label='Adaptive (ACI)')
ax1.axhline(TARGET_ALPHA * 100, color='gray', linestyle=':', label=f'Target α={TARGET_ALPHA}')
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel('Flag rate (%)'); ax1.set_title('Model Decay: Rigid vs Adaptive Flag Rate Across Windows')
ax1.legend(); ax1.grid(True, alpha=0.3)

# 2. Val error rate (concept drift)
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(x, decay_df['val_error_rate'] * 100, color='salmon', alpha=0.8)
ax2.axhline(TARGET_ALPHA * 100, color='gray', linestyle=':')
ax2.set_xticks(x); ax2.set_xticklabels(labels)
ax2.set_ylabel('Val error rate (%)'); ax2.set_title('Concept Drift (Val Error Rate)')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Adaptive alpha vs target
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(x, decay_df['adaptive_alpha'], 's-', color='steelblue', label='Adaptive α')
ax3.axhline(TARGET_ALPHA, color='gray', linestyle=':', label=f'Target α={TARGET_ALPHA}')
ax3.set_xticks(x); ax3.set_xticklabels(labels)
ax3.set_ylabel('Alpha'); ax3.set_title('ACI Alpha Correction per Window')
ax3.legend(); ax3.grid(True, alpha=0.3)

plt.suptitle('Model Decay & Adaptive Conformal Inference', fontsize=13, fontweight='bold')
plt.savefig('/workspace/checkpoints/model_decay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_decay.png')

In [ ]:
# ── Wallet score aggregation strategy comparison ──────────────────────────────
# max, mean, and p90 aggregation may produce meaningfully different flag sets.
# High Jaccard / rank-corr → choice is stable; low → motivates principled selection.
# Uses first rolling window val/test wallets for evaluation.

if 'combined_tx_score' not in df_flat.columns:
  raise RuntimeError('Run cell 18 first to compute combined_tx_score in df_flat.')

def _conformal_flags_simple(cal_s, test_s, alpha):
  thr = np.quantile(cal_s, 1 - alpha)
  return (test_s >= thr).astype(int)

_AGG_FUNS = {
  'max':  lambda x: x.max(),
  'mean': lambda x: x.mean(),
  'p90':  lambda x: np.percentile(x, 90),
}

_agg_flags  = {}
_agg_scores = {}

for _agg_name, _fn in _AGG_FUNS.items():
  _ws    = df_flat.groupby('wallet')['combined_tx_score'].agg(_fn)
  _cal_s = _ws[_ws.index.isin(val_wallets)].values
  _tst_s = _ws[_ws.index.isin(test_wallets)].values
  if len(_cal_s) == 0 or len(_tst_s) == 0:
      print(f'WARNING: {_agg_name} — empty cal or test set, skipping')
      continue
  _agg_flags[_agg_name]  = _conformal_flags_simple(_cal_s, _tst_s, ALPHA)
  _agg_scores[_agg_name] = _tst_s

if not _agg_flags:
  raise RuntimeError('No aggregation strategies produced valid scores. Check val_wallets/test_wallets.')

_base_fl = _agg_flags['max']
_base_sc = _agg_scores['max']

agg_comp_rows = []
for _name, _fl in _agg_flags.items():
  _inter = int(np.sum((_fl == 1) & (_base_fl == 1)))
  _union = int(np.sum((_fl == 1) | (_base_fl == 1)))
  _jac   = _inter / _union if _union > 0 else 1.0
  _rho   = float(spearmanr(_agg_scores[_name], _base_sc).statistic)
  agg_comp_rows.append({
      'aggregation':      _name,
      'flagged':          int(_fl.sum()),
      'flag_rate_pct':    round(float(_fl.mean() * 100), 2),
      'jaccard_vs_max':   round(_jac, 4),
      'rank_corr_vs_max': round(_rho, 4),
  })

wallet_agg_df = pd.DataFrame(agg_comp_rows)
print('Wallet aggregation strategy comparison:')
print(wallet_agg_df.to_string(index=False))
print('\nInterpretation: Jaccard < 0.8 or rank_corr < 0.9 means aggregation choice '
    'materially changes which wallets are flagged.')

In [ ]:
# ── Final output ──────────────────────────────────────────────────────────────
df_flat['if_tx_score'] = if_tx_score
df_flat['tr_tx_score'] = tr_tx_score

# ── eval_phase: academic walk-forward label ───────────────────────────────────
# ONLY test_wallets have statistically valid conformal p-values (ICP guarantee).
# Train/val wallets get backfilled p-values as operational heuristics only.
test_mask_w = wallet_scores['wallet'].isin(test_wallets)
wallet_scores['eval_phase'] = np.where(
  test_mask_w, 'Test (Out-of-Sample)', 'Train (In-Sample)')

# ── Backfill conformal_p for Train (In-Sample) wallets ───────────────────────
# Uses cal_scores calibration distribution. Valid as risk heuristic ONLY —
# not a conformal guarantee. Filter to eval_phase=='Test' for paper metrics.
nan_mask = wallet_scores['conformal_p'].isna()
if nan_mask.any():
  nan_scores = wallet_scores.loc[nan_mask, 'raw_score'].values
  fill_pvals = np.array([
      (np.sum(cal_scores >= s) + 1) / (len(cal_scores) + 1)
      for s in nan_scores
  ])
  wallet_scores.loc[nan_mask, 'conformal_p']    = fill_pvals
  wallet_scores.loc[nan_mask, 'conformal_flag'] = (fill_pvals <= ALPHA).astype(int)

# ── risk_score and risk_tier (no NaNs after backfill) ────────────────────────
wallet_scores['risk_score'] = -np.log10(
  np.clip(wallet_scores['conformal_p'], 1e-12, 1.0))
wallet_scores['risk_tier'] = pd.cut(
  wallet_scores['risk_score'],
  bins=[-np.inf, 1, 2, 3, np.inf],
  labels=['low', 'medium', 'high', 'critical'])

# ── Diagnostics ───────────────────────────────────────────────────────────────
print('Wallet scores shape :', wallet_scores.shape)
print('NaN conformal_p     :', wallet_scores['conformal_p'].isna().sum())
print('eval_phase counts\n', wallet_scores['eval_phase'].value_counts())

print('\nTop 20 wallets by raw_score:')
show = ['wallet', 'raw_score', 'if_score', 'tr_score',
      'conformal_p', 'conformal_flag', 'risk_tier', 'eval_phase', 'n_tx']
show = [c for c in show if c in wallet_scores.columns]
print(wallet_scores.nlargest(20, 'raw_score')[show].to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
import os
CKPT = '/workspace/checkpoints/'
os.makedirs(CKPT, exist_ok=True)
wallet_scores.to_parquet(CKPT + 'wallet_scores_final.parquet', index=False)
df_flat.to_parquet(CKPT + 'transactions_scored.parquet', index=False)
print('Saved: wallet_scores_final.parquet, transactions_scored.parquet')

In [ ]:
print(wallet_scores['risk_tier'].value_counts())
print(wallet_scores['conformal_p'].describe())

In [ ]:
# ── External baselines: LOF, One-Class SVM, MLP Autoencoder ──────────────────
# Compared to the baseline (IF + Transformer AE) via wallet-level Jaccard
# and rank correlation on the test set.  No labels required.

from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm       import OneClassSVM

# ── LOF ──────────────────────────────────────────────────────────────────────
print('Fitting LOF ...')
_lof = LocalOutlierFactor(n_neighbors=20, contamination='auto',
                          novelty=True, n_jobs=-1)
_rng_lof = np.random.default_rng(42)
_lof_n = train_mask_tx.sum()
_lof_idx = _rng_lof.choice(_lof_n, size=min(25000, _lof_n), replace=False)
_lof.fit(X_tx_norm[train_mask_tx][_lof_idx])
_lof_raw        = -_lof.decision_function(X_tx_norm)
_lof_train_sort = np.sort(_lof_raw[train_mask_tx])
_lof_score      = np.searchsorted(_lof_train_sort, _lof_raw, side='right') / len(_lof_train_sort)
lof_wallet_score = (pd.Series(_lof_score, index=df_flat.index)
                    .groupby(df_flat['wallet']).max()
                    .rename('lof_score'))

# ── One-Class SVM (subsample — OCSVM doesn't scale to 500 k rows) ────────────
print('Fitting OCSVM (15 k train sample) ...')
_rng_oc  = np.random.default_rng(42)
_oc_n    = train_mask_tx.sum()
_oc_idx  = _rng_oc.choice(_oc_n, size=min(15000, _oc_n), replace=False)
_ocsvm   = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
_ocsvm.fit(X_tx_norm[train_mask_tx][_oc_idx])
_ocsvm_raw        = -_ocsvm.decision_function(X_tx_norm)
_ocsvm_train_sort = np.sort(_ocsvm_raw[train_mask_tx])
_ocsvm_score      = np.searchsorted(_ocsvm_train_sort, _ocsvm_raw, side='right') / len(_ocsvm_train_sort)
ocsvm_wallet_score = (pd.Series(_ocsvm_score, index=df_flat.index)
                      .groupby(df_flat['wallet']).max()
                      .rename('ocsvm_score'))

# ── MLP Autoencoder ───────────────────────────────────────────────────────────
print('Fitting MLP AE ...')
class MlpAE(nn.Module):
    def __init__(self, n_feat, hidden=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_feat, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 32),     nn.ReLU())
        self.dec = nn.Sequential(nn.Linear(32, hidden),     nn.ReLU(),
                                 nn.Linear(hidden, n_feat))
    def forward(self, x):
        return self.dec(self.enc(x))

_mlp = MlpAE(X_tx_norm.shape[1]).to(device)
_mlp_opt = torch.optim.Adam(_mlp.parameters(), lr=1e-3)
_mlp_dl  = DataLoader(TensorDataset(torch.from_numpy(X_tx_norm[train_mask_tx])),
                      batch_size=256, shuffle=True)
_mlp.train()
for _ep in range(6):
    _tot = 0
    for (_xb,) in _mlp_dl:
        _xb  = _xb.to(device)
        _l   = nn.functional.mse_loss(_mlp(_xb), _xb)
        _mlp_opt.zero_grad(); _l.backward(); _mlp_opt.step()
        _tot += _l.item()
    print(f'  MLP AE epoch {_ep+1}/6  loss={_tot/len(_mlp_dl):.6f}')

_mlp.eval()
_mlp_err = []
with torch.no_grad():
    for _i in range(0, len(X_tx_norm), 1024):
        _xb = torch.from_numpy(X_tx_norm[_i:_i+1024]).to(device)
        _mlp_err.append(((_mlp(_xb) - _xb)**2).mean(dim=1).cpu().numpy())
_mlp_raw        = np.concatenate(_mlp_err)
_mlp_train_sort = np.sort(_mlp_raw[train_mask_tx])
_mlp_score      = np.searchsorted(_mlp_train_sort, _mlp_raw, side='right') / len(_mlp_train_sort)
mlp_wallet_score = (pd.Series(_mlp_score, index=df_flat.index)
                    .groupby(df_flat['wallet']).max()
                    .rename('mlp_score'))

# ── Wallet-level comparison ───────────────────────────────────────────────────
_comp_ws = pl.from_pandas(wallet_scores[['wallet', 'raw_score']])
for _df_ext in [lof_wallet_score, ocsvm_wallet_score, mlp_wallet_score]:
    _comp_ws = _comp_ws.join(pl.from_pandas(_df_ext.reset_index()), on='wallet', how='left')
_comp_ws = _comp_ws.to_pandas().rename(columns={'raw_score': 'baseline'})

_val_m  = _comp_ws['wallet'].isin(val_wallets).values
_test_m = _comp_ws['wallet'].isin(test_wallets).values

_base_cal  = _comp_ws.loc[_val_m,  'baseline'].values
_base_test = _comp_ws.loc[_test_m, 'baseline'].values
_base_fl   = (_base_test >= np.quantile(_base_cal, 1 - ALPHA)).astype(int)

_ext_rows = []
for _det in ['baseline', 'lof_score', 'ocsvm_score', 'mlp_score']:
    _cal_s  = _comp_ws.loc[_val_m,  _det].fillna(0).values
    _test_s = _comp_ws.loc[_test_m, _det].fillna(0).values
    _fl     = (_test_s >= np.quantile(_cal_s, 1 - ALPHA)).astype(int)
    _inter  = int(np.sum((_fl == 1) & (_base_fl == 1)))
    _union  = int(np.sum((_fl == 1) | (_base_fl == 1)))
    _jac    = _inter / _union if _union > 0 else 1.0
    _rho    = float(spearmanr(_test_s, _base_test).correlation)
    _ext_rows.append({'detector': _det,
                      'flagged': int(_fl.sum()),
                      'flag_rate_pct': round(float(_fl.mean() * 100), 2),
                      'jaccard_vs_baseline': round(_jac, 4),
                      'rank_corr_vs_baseline': round(_rho, 4)})

external_baselines_df = pd.DataFrame(_ext_rows)
print('\nExternal baseline comparison (wallet-level):')
print(external_baselines_df.to_string(index=False))
print('\nInterpretation: high Jaccard/rank-corr with baseline → consistent signal;'
      ' low → detectors disagree on which wallets are anomalous.')


In [ ]:
# ── 2. Ablation: component contribution vs baseline ───────────────────────────

def conformal_flags(cal_scores, test_scores, alpha):
    threshold = np.quantile(cal_scores, 1 - alpha)
    pvals = np.array([
        (np.sum(cal_scores >= s) + 1) / (len(cal_scores) + 1)
        for s in test_scores
    ])
    flags = (pvals <= alpha).astype(int)
    return threshold, pvals, flags

val_mask_w  = wallet_scores['wallet'].isin(val_wallets).values
test_mask_w = wallet_scores['wallet'].isin(test_wallets).values

score_sets = {
    'baseline': wallet_scores['raw_score'].values,
    'if_only': wallet_scores['if_score'].values,
    'tr_only': wallet_scores['tr_score'].values,
}

_, _, baseline_test_flags = conformal_flags(
    score_sets['baseline'][val_mask_w],
    score_sets['baseline'][test_mask_w],
    ALPHA,
)

rows = []
for name, scores in score_sets.items():
    thr, pvals, flags = conformal_flags(scores[val_mask_w], scores[test_mask_w], ALPHA)

    inter = np.sum((flags == 1) & (baseline_test_flags == 1))
    union = np.sum((flags == 1) | (baseline_test_flags == 1))
    jaccard = inter / union if union > 0 else 1.0

    rows.append({
        'run': name,
        'threshold': float(thr),
        'flagged_wallets': int(flags.sum()),
        'flag_rate_pct': float(flags.mean() * 100),
        'jaccard_vs_baseline': float(jaccard),
    })

ablation_results = pd.DataFrame(rows)
base_rate = float(ablation_results.loc[ablation_results['run'] == 'baseline', 'flag_rate_pct'].iloc[0])
ablation_results['delta_rate_vs_baseline'] = ablation_results['flag_rate_pct'] - base_rate

print('Ablation results (wallet-level conformal flags):')
print(ablation_results.sort_values('run').to_string(index=False))

In [ ]:
# ── Bootstrap confidence intervals for ablation Jaccard ───────────────────────
# Adds 95 % CI to each ablation variant to quantify estimation uncertainty.

_rng_bs = np.random.default_rng(42)
_N_BOOT = 1000

_val_w_arr  = wallet_scores['wallet'].isin(val_wallets).values
_test_w_arr = wallet_scores['wallet'].isin(test_wallets).values

def _bootstrap_jaccard(score_arr, base_arr, val_m, test_m, alpha, n_boot, rng):
    _cal_s  = score_arr[val_m];  _tst_s  = score_arr[test_m]
    _bcal_s = base_arr[val_m];   _btst_s = base_arr[test_m]
    _jacs   = []
    for _ in range(n_boot):
        _bi  = rng.choice(len(_tst_s), size=len(_tst_s), replace=True)
        _fl  = (_tst_s[_bi]  >= np.quantile(_cal_s,  1 - alpha)).astype(int)
        _bfl = (_btst_s[_bi] >= np.quantile(_bcal_s, 1 - alpha)).astype(int)
        _i   = np.sum((_fl == 1) & (_bfl == 1))
        _u   = np.sum((_fl == 1) | (_bfl == 1))
        _jacs.append(_i / _u if _u > 0 else 1.0)
    _j = np.array(_jacs)
    return float(np.mean(_j)), float(np.percentile(_j, 2.5)), float(np.percentile(_j, 97.5))

_baseline_arr = score_sets['baseline']
bs_rows = []
for _name, _scores in score_sets.items():
    _m, _lo, _hi = _bootstrap_jaccard(_scores, _baseline_arr,
                                       _val_w_arr, _test_w_arr,
                                       ALPHA, _N_BOOT, _rng_bs)
    bs_rows.append({'run': _name,
                    'jaccard_mean': round(_m,  4),
                    'ci_2.5':       round(_lo, 4),
                    'ci_97.5':      round(_hi, 4),
                    'ci_width':     round(_hi - _lo, 4)})

bootstrap_ci_df = pd.DataFrame(bs_rows)
print(f'Bootstrap Jaccard CIs (n_boot={_N_BOOT}, 95 % CI):')
print(bootstrap_ci_df.to_string(index=False))


In [ ]:
# ── 3. Sensitivity: stability across hyperparameter ranges ───────────────────

w_if_grid = [0.30, 0.40, 0.50, 0.60, 0.70]
alpha_grid = [0.005, 0.01, 0.02]

if_all = wallet_scores['if_score'].values
tr_all = wallet_scores['tr_score'].values

baseline_test_scores = score_sets['baseline'][test_mask_w]

sens_rows = []# ── Full-feature permutation null: shuffle features + retrain ────────────────
  # Sequential (n_jobs=1) — parallel GPU jobs OOM on 24 GiB with AE loaded.
  # 10 runs × 4 epochs × 40% subsample is fast enough sequentially.

  import torch
  from sklearn.preprocessing import QuantileTransformer
  from joblib import Parallel, delayed

  if 'score_sets' not in globals() or 'wallet_scores' not in globals():
      raise RuntimeError('Run baseline + ablation cells first (`score_sets` / `wallet_scores` missing).')

  _struct_rng = np.random.default_rng(42)
  N_STRUCT_PERM = 10          # p-value resolution 1/11 ≈ 0.09
  PERM_SUBSAMPLE = 0.40       # 40% of train data per run
  STRUCT_PERM_AE_EPOCHS = int(min(TR_CFG.get('epochs', 10), 4)) if 'TR_CFG' in globals() else 4

  _orig_val_mask = wallet_scores['wallet'].isin(val_wallets).to_numpy()
  _orig_test_mask = wallet_scores['wallet'].isin(test_wallets).to_numpy()

  _orig_val_scores = score_sets['baseline'][_orig_val_mask]
  _orig_test_scores = score_sets['baseline'][_orig_test_mask]
  _orig_flags = (_orig_test_scores >= np.quantile(_orig_val_scores, 1 - ALPHA)).astype(int)
  _orig_tail_contrast = float(np.quantile(_orig_test_scores, 0.99) - np.quantile(_orig_test_scores, 0.50))


  def _jaccard_local(a, b):
      i = np.sum((a == 1) & (b == 1))
      u = np.sum((a == 1) | (b == 1))
      return float(i / u) if u > 0 else 1.0


  def _run_perm(seed):
      rng = np.random.default_rng(seed)

      _train_idx = np.where(train_mask_tx)[0]
      _sub_idx = rng.choice(_train_idx, size=int(len(_train_idx) * PERM_SUBSAMPLE), replace=False)
      _sub_mask = np.zeros(len(train_mask_tx), dtype=bool)
      _sub_mask[_sub_idx] = True

      _X_perm = X_tx.copy()
      for _j in range(_X_perm.shape[1]):
          _X_perm[:, _j] = rng.permutation(_X_perm[:, _j])

      _if_model = IsolationForest(
          n_estimators=int(IF_CFG.get('n_estimators', 300)) if 'IF_CFG' in globals() else 300,
          max_samples=IF_CFG.get('max_samples', 256) if 'IF_CFG' in globals() else 256,
          contamination=IF_CFG.get('contamination', 0.005) if 'IF_CFG' in globals() else 0.005,
          random_state=42,
      )
      _if_model.fit(_X_perm[_sub_mask])
      # Reduce internal GPU scorer batch size to fit in available VRAM (~84 MiB free)
      if hasattr(_if_model, '_gpu_scorer') and _if_model._gpu_scorer is not None:
          _if_model._gpu_scorer.batch_size = 10_000
      # Score in batches to avoid OOM (full X_perm = ~264 MiB on GPU)
      _BATCH = 100_000
      _if_raw = np.empty(len(_X_perm), dtype=np.float32)
      for _s in range(0, len(_X_perm), _BATCH):
          _e = min(_s + _BATCH, len(_X_perm))
          _if_raw[_s:_e] = -_if_model.decision_function(_X_perm[_s:_e])
      _if_sc = np.searchsorted(np.sort(_if_raw[_sub_mask]), _if_raw, side='right') / _sub_mask.sum()

      _scaler_p = QuantileTransformer(output_distribution='uniform', n_quantiles=1000, random_state=42).fit(_X_perm[_sub_mask])
      _X_perm_norm = _scaler_p.transform(_X_perm).astype(np.float32)

      _perm_cfg = dict(TR_CFG) if 'TR_CFG' in globals() else {}
      _perm_cfg['epochs'] = STRUCT_PERM_AE_EPOCHS
      _perm_device = torch.device('cpu')  # GPU full from main models; CPU for null retrains

      _tr_model_p, _, _ = train_tx_ae(
          _X_perm_norm[_sub_mask],
          _X_perm_norm[val_mask_tx],
          _perm_cfg,
          _perm_device,
      )
      _tr_raw, _ = score_tx_ae(_tr_model_p, _X_perm_norm, _perm_device)
      _tr_sc = np.searchsorted(np.sort(_tr_raw[_sub_mask]), _tr_raw, side='right') / _sub_mask.sum()

      # Free GPU memory before next run
      del _tr_model_p
      torch.cuda.empty_cache()

      _comb_tx = W_IF * _if_sc + W_TR * _tr_sc
      _perm_wallet = (
          pd.DataFrame({'wallet': df_flat['wallet'].values, 'score': _comb_tx})
          .groupby('wallet')['score']
          .max()
          .reindex(wallet_scores['wallet'])
          .fillna(0.0)
          .to_numpy(dtype=float)
      )

      _perm_val = _perm_wallet[_orig_val_mask]
      _perm_test = _perm_wallet[_orig_test_mask]
      _perm_flags = (_perm_test >= np.quantile(_perm_val, 1 - ALPHA)).astype(int)
      _perm_tail = float(np.quantile(_perm_test, 0.99) - np.quantile(_perm_test, 0.50))

      return {
          'tail_contrast': _perm_tail,
          'jaccard_vs_original_flags': _jaccard_local(_perm_flags, _orig_flags),
          'flag_rate_pct': float(_perm_flags.mean() * 100),
      }


  print(f'Running {N_STRUCT_PERM} permutation retrains (n_jobs=1, {int(PERM_SUBSAMPLE*100)}% subsample)...')
  _perm_results = Parallel(n_jobs=1, verbose=5)(
      delayed(_run_perm)(seed) for seed in range(N_STRUCT_PERM)
  )

  _struct_rows = [{'perm_run': i + 1, **r} for i, r in enumerate(_perm_results)]
  _struct_tail_null = [r['tail_contrast'] for r in _perm_results]

  struct_perm_df = pd.DataFrame(_struct_rows)

  _k_struct = int(np.sum(np.array(_struct_tail_null) >= _orig_tail_contrast))
  _p_struct = float((_k_struct + 1) / (N_STRUCT_PERM + 1))

  struct_perm_summary = pd.DataFrame([{
      'original_tail_contrast': round(_orig_tail_contrast, 4),
      'null_tail_contrast_mean': round(float(np.mean(_struct_tail_null)), 4),
      'null_tail_contrast_std': round(float(np.std(_struct_tail_null)), 4),
      'k_null_ge_original': _k_struct,
      'perm_p': round(_p_struct, 4),
      'significant': _p_struct < 0.05,
  }])

  print('\nFeature-permutation null results:')
  print(struct_perm_summary.to_string(index=False))
  print('\nPer-run diagnostics:')
  print(struct_perm_df.to_string(index=False))
  print('\nInterpretation: significant means the original model has stronger anomaly-tail\n'
        'structure than shuffled-feature retrains, supporting non-random latent signal.')
for w_if in w_if_grid:
    w_tr = 1.0 - w_if
    mixed_scores = w_if * if_all + w_tr * tr_all

    for alpha in alpha_grid:
        _, _, flags = conformal_flags(mixed_scores[val_mask_w], mixed_scores[test_mask_w], alpha)

        inter = np.sum((flags == 1) & (baseline_test_flags == 1))
        union = np.sum((flags == 1) | (baseline_test_flags == 1))
        jaccard = inter / union if union > 0 else 1.0

        rho = spearmanr(mixed_scores[test_mask_w], baseline_test_scores).correlation
        if np.isnan(rho):
            rho = 0.0

        sens_rows.append({
            'w_if': float(w_if),
            'w_tr': float(w_tr),
            'alpha': float(alpha),
            'flag_rate_pct': float(flags.mean() * 100),
            'jaccard_vs_baseline': float(jaccard),
            'rank_corr_vs_baseline': float(rho),
        })

sensitivity_results = pd.DataFrame(sens_rows)
print('Sensitivity sweep:')
print(sensitivity_results.sort_values(['alpha', 'w_if']).to_string(index=False))

In [ ]:
# ── Statistical significance tests ──────────────────────────────────────────
# McNemar's test: are the disagreements between baseline and each comparison
# detector asymmetric?  This is the correct paired test for binary detectors —
# it asks whether one detector consistently flags wallets the other misses,
# beyond what chance would produce.
#
# Covers two groups:
#   (A) Ablation variants  — baseline vs if_only, tr_only
#   (B) External baselines — baseline vs LOF, OCSVM, MLP AE
# Both groups use the same conformal threshold (alpha=ALPHA on val wallets).

from statsmodels.stats.contingency_tables import mcnemar

_val_w_arr  = wallet_scores['wallet'].isin(val_wallets).values
_test_w_arr = wallet_scores['wallet'].isin(test_wallets).values

def _get_flags(score_arr, val_m, test_m, alpha):
    thr = np.quantile(score_arr[val_m], 1 - alpha)
    return (score_arr[test_m] >= thr).astype(int)

_base_fl = _get_flags(score_sets['baseline'], _val_w_arr, _test_w_arr, ALPHA)

# ── Build combined comparison dict: ablation + external baselines ─────────────
_all_comparisons = dict(score_sets)   # baseline, if_only, tr_only

if '_comp_ws' in globals() and isinstance(_comp_ws, pd.DataFrame):
    _ext_score_cols = {
        'lof':   'lof_score',
        'ocsvm': 'ocsvm_score',
        'mlp':   'mlp_score',
    }
    for _ext_name, _col in _ext_score_cols.items():
        if _col in _comp_ws.columns:
            # Align to wallet_scores index order
            _ext_aligned = (
                _comp_ws[['wallet', _col]]
                .drop_duplicates('wallet')
                .set_index('wallet')
                .reindex(wallet_scores['wallet'])
                [_col]
                .fillna(0.0)
                .values
            )
            _all_comparisons[_ext_name] = _ext_aligned

# ── McNemar test ──────────────────────────────────────────────────────────────
_mcnemar_rows = []
print('McNemar test (baseline vs comparison detector, test-set wallet flags):')
print(f'  {"comparison":<32} {"group":<10} {"chi2":>8} {"p-value":>12} {"sig?":>6}')
print('  ' + '-' * 74)

_group_map = {'if_only': 'ablation', 'tr_only': 'ablation',
              'lof': 'external', 'ocsvm': 'external', 'mlp': 'external'}

for _nm, _scores in _all_comparisons.items():
    if _nm == 'baseline':
        continue
    _cmp_fl = _get_flags(_scores, _val_w_arr, _test_w_arr, ALPHA)
    _b = int(np.sum((_base_fl == 1) & (_cmp_fl == 0)))  # baseline-only
    _c = int(np.sum((_base_fl == 0) & (_cmp_fl == 1)))  # comparator-only
    if _b + _c == 0:
        print(f'  {"baseline vs " + _nm:<32} {"n/a":<10} {"—":>8} {"—":>12} {"—":>6}')
        print(f'    no disagreements between detectors')
        continue
    _res  = mcnemar([[0, _b], [_c, 0]], exact=False, correction=True)
    _sig  = 'YES' if _res.pvalue < 0.05 else 'no'
    _grp  = _group_map.get(_nm, 'ablation')
    print(f'  {"baseline vs " + _nm:<32} {_grp:<10} {_res.statistic:>8.3f} {_res.pvalue:>12.4e} {_sig:>6}')
    print(f'    baseline-only: {_b}  |  {_nm}-only: {_c}')
    _mcnemar_rows.append({
        'comparison': f'baseline vs {_nm}',
        'group': _grp,
        'baseline_only_flags': _b,
        'comparator_only_flags': _c,
        'chi2': round(float(_res.statistic), 4),
        'pvalue': float(_res.pvalue),
        'significant_p05': _res.pvalue < 0.05,
    })

mcnemar_results = pd.DataFrame(_mcnemar_rows)

# ── Permutation test: is the observed Jaccard above chance? ──────────────────
_rng_p  = np.random.default_rng(42)
_N_PERM = 500

_base_val = score_sets['baseline'][_val_w_arr]
_base_tst = score_sets['baseline'][_test_w_arr]
_base_fl2 = (_base_tst >= np.quantile(_base_val, 1 - ALPHA)).astype(int)

def _obs_jaccard(fl_a, fl_b):
    _i = np.sum((fl_a == 1) & (fl_b == 1))
    _u = np.sum((fl_a == 1) | (fl_b == 1))
    return _i / _u if _u > 0 else 1.0

perm_rows = []
for _nm, _sc in _all_comparisons.items():
    if _nm == 'baseline':
        continue
    _val_s  = _sc[_val_w_arr]
    _tst_s  = _sc[_test_w_arr]
    _obs_fl = (_tst_s >= np.quantile(_val_s, 1 - ALPHA)).astype(int)
    _obs_j  = _obs_jaccard(_obs_fl, _base_fl2)
    _null_j = np.empty(_N_PERM, dtype=float)
    for _b in range(_N_PERM):
        _shuf   = _rng_p.permutation(_tst_s)
        _fl_s   = (_shuf >= np.quantile(_val_s, 1 - ALPHA)).astype(int)
        _null_j[_b] = _obs_jaccard(_fl_s, _base_fl2)
    _k    = int(np.sum(_null_j >= _obs_j))
    _pval = float((_k + 1) / (_N_PERM + 1))   # add-one correction
    perm_rows.append({
        'comparison':       f'baseline vs {_nm}',
        'group':            _group_map.get(_nm, 'ablation'),
        'observed_jaccard': round(_obs_j, 4),
        'null_mean_jaccard':round(float(_null_j.mean()), 4),
        'perm_p':           round(_pval, 4),
        'significant':      _pval < 0.05,
    })

perm_df = pd.DataFrame(perm_rows)
print(f'\nPermutation test for Jaccard (n_perm={_N_PERM}):')
print(perm_df.to_string(index=False))
print('\nNote: permutation p = (k+1)/(n_perm+1) (add-one correction).')
print('      McNemar tests disagreement asymmetry; permutation tests flag-set overlap vs chance.')


In [ ]:
# ── Full-feature permutation null: shuffle features + retrain ────────────────
# Frees main models before running so permutation AE can use GPU.
# Checkpoints each run to /workspace/checkpoints/perm_runs/ — safe to interrupt
# and resume; completed seeds are skipped automatically.

import torch, gc, os, pickle
from sklearn.preprocessing import QuantileTransformer
from joblib import Parallel, delayed

if 'score_sets' not in globals() or 'wallet_scores' not in globals():
  raise RuntimeError('Run baseline + ablation cells first.')

CKPT_PERM = '/workspace/checkpoints/perm_runs/'
os.makedirs(CKPT_PERM, exist_ok=True)

# ── Free main models to reclaim GPU for permutation AE retrains ──────────────
for _var in ['tr_model', 'if_model']:
  if _var in globals():
      del globals()[_var]
      print(f'Freed {_var} from GPU memory.')
torch.cuda.empty_cache()
gc.collect()
print(f'GPU memory freed. Available: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GiB')

_struct_rng = np.random.default_rng(42)
N_STRUCT_PERM = 10
PERM_SUBSAMPLE = 0.40
STRUCT_PERM_AE_EPOCHS = int(min(TR_CFG.get('epochs', 10), 4)) if 'TR_CFG' in globals() else 4

_orig_val_mask  = wallet_scores['wallet'].isin(val_wallets).to_numpy()
_orig_test_mask = wallet_scores['wallet'].isin(test_wallets).to_numpy()

_orig_val_scores    = score_sets['baseline'][_orig_val_mask]
_orig_test_scores   = score_sets['baseline'][_orig_test_mask]
_orig_flags         = (_orig_test_scores >= np.quantile(_orig_val_scores, 1 - ALPHA)).astype(int)
_orig_tail_contrast = float(np.quantile(_orig_test_scores, 0.99) - np.quantile(_orig_test_scores, 0.50))


def _jaccard_local(a, b):
  i = np.sum((a == 1) & (b == 1))
  u = np.sum((a == 1) | (b == 1))
  return float(i / u) if u > 0 else 1.0


def _run_perm(seed):
  _ckpt = os.path.join(CKPT_PERM, f'perm_seed_{seed}.pkl')
  if os.path.exists(_ckpt):
      print(f'  [seed {seed}] Resuming from checkpoint.')
      with open(_ckpt, 'rb') as f:
          return pickle.load(f)

  rng = np.random.default_rng(seed)

  _train_idx = np.where(train_mask_tx)[0]
  _sub_idx   = rng.choice(_train_idx, size=int(len(_train_idx) * PERM_SUBSAMPLE), replace=False)
  _sub_mask  = np.zeros(len(train_mask_tx), dtype=bool)
  _sub_mask[_sub_idx] = True

  _X_perm = X_tx.copy()
  for _j in range(_X_perm.shape[1]):
      _X_perm[:, _j] = rng.permutation(_X_perm[:, _j])

  _if_model = IsolationForest(
      n_estimators=int(IF_CFG.get('n_estimators', 300)) if 'IF_CFG' in globals() else 300,
      max_samples=IF_CFG.get('max_samples', 256) if 'IF_CFG' in globals() else 256,
      contamination=IF_CFG.get('contamination', 0.005) if 'IF_CFG' in globals() else 0.005,
      random_state=42,
  )
  _if_model.fit(_X_perm[_sub_mask])
  if hasattr(_if_model, '_gpu_scorer') and _if_model._gpu_scorer is not None:
      _if_model._gpu_scorer.batch_size = 10_000
  _BATCH = 100_000
  _if_raw = np.empty(len(_X_perm), dtype=np.float32)
  for _s in range(0, len(_X_perm), _BATCH):
      _e = min(_s + _BATCH, len(_X_perm))
      _if_raw[_s:_e] = -_if_model.decision_function(_X_perm[_s:_e])
  _if_sc = np.searchsorted(np.sort(_if_raw[_sub_mask]), _if_raw, side='right') / _sub_mask.sum()

  _scaler_p    = QuantileTransformer(output_distribution='uniform', n_quantiles=1000, random_state=42).fit(_X_perm[_sub_mask])
  _X_perm_norm = _scaler_p.transform(_X_perm).astype(np.float32)

  _perm_cfg           = dict(TR_CFG) if 'TR_CFG' in globals() else {}
  _perm_cfg['epochs'] = STRUCT_PERM_AE_EPOCHS

  _tr_model_p, _, _ = train_tx_ae(
      _X_perm_norm[_sub_mask],
      _X_perm_norm[val_mask_tx],
      _perm_cfg,
      device,                      # GPU — main models freed above
  )
  _tr_raw, _ = score_tx_ae(_tr_model_p, _X_perm_norm, device)
  _tr_sc = np.searchsorted(np.sort(_tr_raw[_sub_mask]), _tr_raw, side='right') / _sub_mask.sum()

  del _tr_model_p
  torch.cuda.empty_cache()

  _comb_tx = W_IF * _if_sc + W_TR * _tr_sc
  _perm_wallet = (
      pd.DataFrame({'wallet': df_flat['wallet'].values, 'score': _comb_tx})
      .groupby('wallet')['score']
      .max()
      .reindex(wallet_scores['wallet'])
      .fillna(0.0)
      .to_numpy(dtype=float)
  )

  _perm_val   = _perm_wallet[_orig_val_mask]
  _perm_test  = _perm_wallet[_orig_test_mask]
  _perm_flags = (_perm_test >= np.quantile(_perm_val, 1 - ALPHA)).astype(int)
  _perm_tail  = float(np.quantile(_perm_test, 0.99) - np.quantile(_perm_test, 0.50))

  result = {
      'tail_contrast':             _perm_tail,
      'jaccard_vs_original_flags': _jaccard_local(_perm_flags, _orig_flags),
      'flag_rate_pct':             float(_perm_flags.mean() * 100),
  }
  with open(_ckpt, 'wb') as f:
      pickle.dump(result, f)
  print(f'  [seed {seed}] Saved checkpoint.')
  return result


print(f'Running {N_STRUCT_PERM} permutation retrains (n_jobs=1, {int(PERM_SUBSAMPLE*100)}% subsample)...')
_perm_results = Parallel(n_jobs=1, verbose=5)(
  delayed(_run_perm)(seed) for seed in range(N_STRUCT_PERM)
)

_struct_rows      = [{'perm_run': i + 1, **r} for i, r in enumerate(_perm_results)]
_struct_tail_null = [r['tail_contrast'] for r in _perm_results]

struct_perm_df = pd.DataFrame(_struct_rows)

_k_struct = int(np.sum(np.array(_struct_tail_null) >= _orig_tail_contrast))
_p_struct = float((_k_struct + 1) / (N_STRUCT_PERM + 1))

struct_perm_summary = pd.DataFrame([{
  'original_tail_contrast':  round(_orig_tail_contrast, 4),
  'null_tail_contrast_mean': round(float(np.mean(_struct_tail_null)), 4),
  'null_tail_contrast_std':  round(float(np.std(_struct_tail_null)), 4),
  'k_null_ge_original':      _k_struct,
  'perm_p':                  round(_p_struct, 4),
  'significant':             _p_struct < 0.05,
}])

print('\nFeature-permutation null results:')
print(struct_perm_summary.to_string(index=False))
print('\nPer-run diagnostics:')
print(struct_perm_df.to_string(index=False))
print('\nInterpretation: significant means the original model has stronger anomaly-tail\n'
    'structure than shuffled-feature retrains, supporting non-random latent signal.')

In [ ]:
# ── Checkpoint: save permutation null results ────────────────────────────────
import os

CKPT = '/workspace/checkpoints/'
os.makedirs(CKPT, exist_ok=True)

struct_perm_df.to_parquet(CKPT + 'struct_perm_df.parquet', index=False)
struct_perm_summary.to_parquet(CKPT + 'struct_perm_summary.parquet', index=False)
print('Saved: struct_perm_df.parquet, struct_perm_summary.parquet')


In [ ]:
# ── 4. Contamination: baseline detection on synthetic anomalies ──────────────
import torch, joblib

_ckpt    = torch.load('/workspace/checkpoints/ae_model.pt', map_location=device)
_cfg     = _ckpt['cfg']
_cfg_keys = TxTransformerAE.__init__.__code__.co_varnames

tr_model = TxTransformerAE(
  n_features=X_tx.shape[1],
  **{k: v for k, v in _cfg.items() if k in _cfg_keys and k != 'n_features'}
)
tr_model.load_state_dict(_ckpt['state_dict'])
tr_model = tr_model.to(device).eval()

iforest = joblib.load('/workspace/checkpoints/iforest.pkl')
print('Models reloaded.')

rng = np.random.default_rng(42)
feat_idx = {f: i for i, f in enumerate(tx_feature_cols)}

normal_pool = np.where(test_mask_tx & (combined_tx <= np.quantile(combined_tx[test_mask_tx], 0.50)))[0]
if len(normal_pool) == 0:
    normal_pool = np.where(test_mask_tx)[0]

n_syn = int(min(5000, len(normal_pool)))
if n_syn == 0:
    raise RuntimeError('No test transactions available for contamination analysis.')
syn_pick = rng.choice(normal_pool, size=n_syn, replace=False)
X_syn_raw = X_tx[syn_pick].copy()

# Inject synthetic anomalous patterns
for f in ['fee_sol', 'compute_units_consumed', 'fanout_ratio', 'cpi_ratio', 'drain_sol_ratio', 'inner_instructions']:
    if f in feat_idx:
        j = feat_idx[f]
        X_syn_raw[:, j] = X_syn_raw[:, j] * rng.uniform(2.5, 6.0, size=n_syn)

if 'success_flag' in feat_idx:
    X_syn_raw[:, feat_idx['success_flag']] = 0.0

train_std = X_tx[train_mask_tx].std(axis=0) + 1e-6
X_syn_raw += rng.normal(0.0, 1.25 * train_std, size=X_syn_raw.shape)

# Re-score synthetic tx with trained baseline components
if_syn_raw = -iforest.decision_function(X_syn_raw)
# Training-CDF normalisation: reuse sorted training arrays from cells 9/14.
if_syn = np.searchsorted(_if_train_sort, if_syn_raw, side='right') / len(_if_train_sort)

X_syn_norm = scaler.transform(X_syn_raw).astype(np.float32)
tr_syn_raw, _ = score_tx_ae(tr_model, X_syn_norm, device)
tr_syn = np.searchsorted(_tr_train_sort, tr_syn_raw, side='right') / len(_tr_train_sort)

combined_syn = np.maximum(if_syn, tr_syn)

if_thr = np.quantile(if_tx_score[val_mask_tx], 1 - ALPHA)
tr_thr = np.quantile(tr_tx_score[val_mask_tx], 1 - ALPHA)
comb_thr = np.quantile(combined_tx[val_mask_tx], 1 - ALPHA)

contamination_results = pd.DataFrame([
    {'detector': 'if_only', 'threshold': float(if_thr), 'synthetic_detection_rate_pct': float((if_syn >= if_thr).mean() * 100)},
    {'detector': 'tr_only', 'threshold': float(tr_thr), 'synthetic_detection_rate_pct': float((tr_syn >= tr_thr).mean() * 100)},
    {'detector': 'baseline_combined', 'threshold': float(comb_thr), 'synthetic_detection_rate_pct': float((combined_syn >= comb_thr).mean() * 100)},
])

print('Synthetic contamination detection rates:')
print(contamination_results.to_string(index=False))

# Heatmap: archetype × detector
import seaborn as sns
import matplotlib.pyplot as plt

_heat = archetype_detection_df.set_index('archetype')[['IF %', 'TR %', 'Combined %']]
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(_heat, annot=True, fmt='.1f', cmap='YlOrRd', vmin=0, vmax=100,
            linewidths=0.5, ax=ax)
ax.set_title(f'Synthetic Detection Rate (%) by Archetype & Detector\n'
             f'(injection multiplier={INJECT_MULTIPLIER}x, n={n_syn} per archetype)')
ax.set_xlabel('Detector')
ax.set_ylabel('Injected Archetype')
plt.tight_layout()
plt.savefig('archetype_detection_rates.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: archetype_detection_rates.png')

In [ ]:
# ── Interpretability: what did the pipeline flag and why ─────────────────────
# For each flagged test wallet:
#   - Dominant component: IF vs AE (which score drove the flag)
#   - Top AE feature errors: which features were hardest to reconstruct
#   - Shown as a heatmap across top flagged wallets

if 'per_feat_errors' not in globals():
  per_feat_errors = np.load('/workspace/checkpoints/per_feat_errors.npy')

flagged = wallet_scores[
  wallet_scores['wallet'].isin(test_wallets) &
  (wallet_scores['conformal_flag'].fillna(0).astype(int) == 1)
].sort_values('raw_score', ascending=False).head(20).copy()

if len(flagged) == 0:
  print('No flagged test wallets at current alpha.')
else:
  # Top transaction per flagged wallet (highest AE error)
  _tr_col = 'tr_tx_score' if 'tr_tx_score' in df_flat.columns else 'if_tx_score'
  tx_top_idx = df_flat.groupby('wallet')[_tr_col].idxmax()

  feat_matrix = []
  labels = []
  rows = []

  for _, row in flagged.iterrows():
      w = row['wallet']
      tx_idx = tx_top_idx.get(w)
      if tx_idx is None:
          continue

      feat_err = per_feat_errors[int(tx_idx)]
      topk_idx = np.argsort(feat_err)[-5:][::-1]
      dominant = 'IF' if row['if_score'] >= row['tr_score'] else 'AE'

      feat_matrix.append(feat_err)
      labels.append(f"{w[:8]}… [{dominant}] p={row['conformal_p']:.3f}")
      rows.append({
          'wallet':      w[:12] + '…',
          'raw_score':   round(float(row['raw_score']), 4),
          'if_score':    round(float(row['if_score']),  4),
          'tr_score':    round(float(row['tr_score']),  4),
          'conformal_p': round(float(row['conformal_p']), 4) if pd.notna(row['conformal_p']) else np.nan,
          'dominant':    dominant,
          'top_features': ', '.join(tx_feature_cols[i] for i in topk_idx),
      })

  print(pd.DataFrame(rows).to_string(index=False))

  # ── Heatmap: flagged wallets × top features ───────────────────────────────
  feat_mat = np.array(feat_matrix)                         # (n_wallets, n_features)
  top_feat_idx = np.argsort(feat_mat.mean(axis=0))[-15:]  # top 15 features by mean error
  feat_mat_top = feat_mat[:, top_feat_idx]
  feat_names_top = [tx_feature_cols[i] for i in top_feat_idx]

  fig, ax = plt.subplots(figsize=(14, max(4, len(labels) * 0.5)))
  sns.heatmap(feat_mat_top, annot=False, cmap='YlOrRd',
              xticklabels=feat_names_top, yticklabels=labels,
              linewidths=0.3, ax=ax)
  ax.set_title('AE Reconstruction Error by Feature — Top Flagged Wallets', fontweight='bold')
  ax.set_xlabel('Feature'); ax.set_ylabel('Wallet [dominant detector] [conformal p]')
  ax.tick_params(axis='x', rotation=45)
  plt.tight_layout()
  plt.savefig('/workspace/checkpoints/interpretability_heatmap.png', dpi=150, bbox_inches='tight')
  plt.show()
  print('Saved: interpretability_heatmap.png')

In [ ]:
# ── SHAP interpretability (Isolation Forest component) ────────────────────────
# Addresses the black-box concern by explaining which features drive IF anomaly
# scores for top anomalous test transactions.

shap_global_df = pd.DataFrame()
shap_local_top_df = pd.DataFrame()

try:
    import shap
except Exception:
    print('SHAP is not installed in this environment. Run `pip install shap` and re-run this cell.')
else:
    required_vars = ['iforest', 'if_tx_score', 'X_tx', 'test_mask_tx', 'tx_feature_cols', 'df_flat']
    missing_vars = [v for v in required_vars if v not in globals()]
    if missing_vars:
        raise RuntimeError(f'Missing prerequisites for SHAP: {missing_vars}')

    bg_n = int(min(2000, np.sum(train_mask_tx))) if 'train_mask_tx' in globals() else int(min(2000, len(X_tx)))
    ex_n = int(min(300, np.sum(test_mask_tx)))
    if ex_n <= 0:
        raise RuntimeError('No test transactions available for SHAP interpretation.')

    # OPTIMIZED: Use Lance scanner for background sampling if LANCE_PATH exists
    if 'LANCE_PATH' in globals() and os.path.exists(LANCE_PATH):
        print(f'Using Lance scanner to sample {bg_n} background transactions...')
        X_bg = sample_lance_data(LANCE_PATH, bg_n, columns=list(tx_feature_cols)).values.astype(np.float32)
    else:
        rng_shap = np.random.default_rng(42)
        if 'train_mask_tx' in globals() and np.sum(train_mask_tx) > 0:
            train_idx = np.where(train_mask_tx)[0]
        else:
            train_idx = np.arange(len(X_tx))
        bg_idx = rng_shap.choice(train_idx, size=bg_n, replace=False) if len(train_idx) > bg_n else train_idx
        X_bg = X_tx[bg_idx].astype(np.float32)
    X_ex = X_tx[explain_idx].astype(np.float32)

    shap_values = None
    shap_mode = 'tree'

    try:
        explainer = shap.TreeExplainer(iforest)
        shap_values = explainer.shap_values(X_ex)
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
        shap_values = np.asarray(shap_values, dtype=float)
    except Exception:
        shap_mode = 'model_agnostic'

        def _if_anom(x):
            x = np.asarray(x, dtype=np.float32)
            return -iforest.decision_function(x)

        explainer = shap.Explainer(_if_anom, X_bg, feature_names=tx_feature_cols)
        shap_out = explainer(X_ex)
        shap_values = np.asarray(shap_out.values, dtype=float)

    if shap_values.ndim != 2 or shap_values.shape[1] != len(tx_feature_cols):
        raise RuntimeError(f'Unexpected SHAP output shape: {shap_values.shape}')

    # Global IF feature importance
    mean_abs = np.mean(np.abs(shap_values), axis=0)
    shap_global_df = (
        pd.DataFrame({'feature': tx_feature_cols, 'mean_abs_shap': mean_abs})
        .sort_values('mean_abs_shap', ascending=False)
        .reset_index(drop=True)
    )

    print(f'SHAP mode: {shap_mode}')
    print('Top IF feature drivers by mean |SHAP|:')
    print(shap_global_df.head(15).to_string(index=False))

    # Local explanations for top anomalous transactions
    top_feat = shap_global_df.head(12)['feature'].tolist()
    top_feat_idx = [tx_feature_cols.index(f) for f in top_feat]

    shap_local_top_df = pd.DataFrame(
        shap_values[:, top_feat_idx],
        columns=top_feat,
    )
    shap_local_top_df['tx_idx'] = explain_idx
    shap_local_top_df['wallet'] = df_flat.iloc[explain_idx]['wallet'].astype(str).values
    shap_local_top_df['if_score'] = if_tx_score[explain_idx]

    # Seaborn visualisations
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

    sns.barplot(data=shap_global_df.head(15), x='mean_abs_shap', y='feature', color='#4C78A8', ax=axes[0])
    axes[0].set_title('SHAP global importance (Isolation Forest)')
    axes[0].set_xlabel('Mean |SHAP value|')
    axes[0].set_ylabel('Feature')

    hm_n = int(min(20, len(shap_local_top_df)))
    hm = shap_local_top_df[top_feat].head(hm_n).to_numpy(dtype=float)
    row_labels = []
    for w in shap_local_top_df['wallet'].head(hm_n):
        ws = str(w)
        row_labels.append(ws if len(ws) <= 14 else f'{ws[:6]}...{ws[-4:]}')

    sns.heatmap(
        hm,
        cmap='coolwarm',
        center=0.0,
        xticklabels=top_feat,
        yticklabels=row_labels,
        cbar_kws={'label': 'SHAP value'},
        ax=axes[1],
    )
    axes[1].set_title('Local SHAP (top anomalous test transactions)')
    axes[1].set_xlabel('Feature')
    axes[1].set_ylabel('Transaction (wallet)')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    if 'no_label_dashboard' in globals():
        no_label_dashboard['if_shap_global'] = shap_global_df
        no_label_dashboard['if_shap_local_top'] = shap_local_top_df

print('Saved SHAP outputs: `shap_global_df`, `shap_local_top_df` (and dashboard keys if available).')

In [ ]:
# ── Transformer AE attention weights (interpretability) ──────────────────────
# Complements per_feat_errors (what failed to reconstruct) by showing
# *which feature–feature interactions* the encoder was attending to.
#
# Method: call each TransformerEncoderLayer's self_attn directly with
# need_weights=True for the top-N anomalous test transactions, then average
# over heads and transactions.  Result: one F×F heatmap per encoder layer
# showing mean attention from token i → token j.
#
# A second panel ranks the strongest off-diagonal feature pairs, giving a
# readable summary of which cross-feature dependencies drove the anomaly.

_REQUIRED_ATTN = ['tr_model', 'X_tx_norm', 'test_mask_tx',
                  'tx_feature_cols', 'tr_tx_score', 'device']
_miss_attn = [v for v in _REQUIRED_ATTN if v not in globals()]
if _miss_attn:
    raise RuntimeError(f'Missing prerequisites for attention viz: {_miss_attn}')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

N_TOP_ATTN = 10   # top anomalous test transactions to average over

test_idx_attn = np.where(test_mask_tx)[0]
top_order_attn = np.argsort(tr_tx_score[test_mask_tx])[-N_TOP_ATTN:][::-1]
explain_idx_attn = test_idx_attn[top_order_attn]

feat_names = list(tx_feature_cols)
F_dim = len(feat_names)

X_top_attn = torch.from_numpy(X_tx_norm[explain_idx_attn]).to(device)  # (N_TOP, F)

tr_model.eval()
layer_attns = []   # list of np arrays (N_TOP, F, F) — averaged over heads per layer

with torch.no_grad():
    pos = torch.arange(F_dim, device=device)
    h   = tr_model.feat_proj(X_top_attn.unsqueeze(-1)) + tr_model.pos_embed(pos).unsqueeze(0)
    # h: (N_TOP, F, d_model)

    for layer in tr_model.encoder.layers:
        # Extract attention weights with need_weights=True.
        # average_attn_weights=False → (B, nhead, F, F); added in PyTorch 1.11.
        # Fallback: older PyTorch returns (B, F, F) already averaged.
        try:
            _, attn_w = layer.self_attn(
                h, h, h, need_weights=True, average_attn_weights=False
            )
            # attn_w: (B, nhead, F, F)
            if attn_w is not None:
                avg_heads = attn_w.mean(dim=1).cpu().numpy()   # (B, F, F)
            else:
                avg_heads = None
        except TypeError:
            _, attn_w = layer.self_attn(h, h, h, need_weights=True)
            # attn_w: (B, F, F)
            avg_heads = attn_w.cpu().numpy() if attn_w is not None else None

        if avg_heads is not None:
            layer_attns.append(avg_heads.mean(axis=0))   # (F, F) mean over top-N txns

        h = layer(h)   # continue forward pass normally

if not layer_attns:
    print('Attention weights unavailable (need_weights returned None). '
          'Check PyTorch version ≥ 1.9.')
else:
    n_layers = len(layer_attns)
    sns.set_theme(style='white', font_scale=0.85)
    pal = sns.color_palette('rocket_r', as_cmap=True)

    # ── Figure 1: per-layer F×F heatmap ──────────────────────────────────────
    fig1, axes1 = plt.subplots(1, n_layers, figsize=(8 * n_layers, 7),
                               constrained_layout=True)
    if n_layers == 1:
        axes1 = [axes1]

    for li, (ax, attn_map) in enumerate(zip(axes1, layer_attns)):
        mask = np.eye(F_dim, dtype=bool)   # suppress diagonal (self-attention)
        sns.heatmap(
            attn_map,
            ax=ax,
            cmap=pal,
            mask=mask,
            xticklabels=feat_names,
            yticklabels=feat_names,
            linewidths=0,
            vmin=0,
            cbar_kws={'label': 'Mean attention weight', 'shrink': 0.7},
        )
        ax.set_title(
            f'Encoder layer {li + 1}  |  avg over {N_TOP_ATTN} top-anomaly txns',
            fontsize=11, pad=8
        )
        ax.tick_params(axis='x', rotation=90, labelsize=6.5)
        ax.tick_params(axis='y', rotation=0,  labelsize=6.5)
        ax.set_xlabel('Key feature (attended to)', fontsize=9)
        ax.set_ylabel('Query feature (attending from)', fontsize=9)

    fig1.suptitle(
        'Transformer AE — self-attention feature interaction map\n'
        f'(diagonal suppressed; {N_TOP_ATTN} most anomalous test transactions)',
        fontsize=12, y=1.02
    )
    plt.show()

    # ── Figure 2: top off-diagonal feature pairs ──────────────────────────────
    # Average attention across all layers, then rank off-diagonal pairs.
    mean_attn = np.mean(layer_attns, axis=0)          # (F, F)
    np.fill_diagonal(mean_attn, 0.0)

    N_PAIRS = 20
    flat_idx = np.argsort(mean_attn, axis=None)[-N_PAIRS:][::-1]
    rows_i, cols_j = np.unravel_index(flat_idx, mean_attn.shape)

    pair_df = pd.DataFrame({
        'query_feature':  [feat_names[i] for i in rows_i],
        'key_feature':    [feat_names[j] for j in cols_j],
        'mean_attention': mean_attn[rows_i, cols_j],
    })

    fig2, ax2 = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    bar_labels = [f"{r}  →  {k}" for r, k in zip(pair_df['query_feature'], pair_df['key_feature'])]
    colors = sns.color_palette('rocket_r', N_PAIRS)[::-1]
    ax2.barh(range(N_PAIRS), pair_df['mean_attention'].values[::-1], color=colors[::-1])
    ax2.set_yticks(range(N_PAIRS))
    ax2.set_yticklabels(bar_labels[::-1], fontsize=8.5)
    ax2.set_xlabel('Mean attention weight (query → key)', fontsize=10)
    ax2.set_title(
        f'Top {N_PAIRS} cross-feature attention pairs\n'
        '(averaged over encoder layers and top-anomaly transactions)',
        fontsize=11
    )
    ax2.spines[['top', 'right']].set_visible(False)
    plt.show()

    print(f'\nTop {N_PAIRS} attended feature pairs:')
    print(pair_df.to_string(index=False))
    print(
        '\nInterpretation: high attention weight from query Q → key K means the encoder'
        '\nconsiders feature K context when reconstructing Q.  Pairs that consistently'
        '\nappear in top-anomaly transactions but not in normal traffic indicate the'
        '\ncross-feature patterns most associated with anomalous behaviour.'
    )


In [ ]:
# ── No-label evaluation: model error metrics ──────────────────────────────────
from scipy.stats import ks_2samp, wasserstein_distance

def _clean(x):
  x = np.asarray(x, dtype=float)
  return x[np.isfinite(x)]

def _drift_ks(ref, cur):
  r, c = _clean(ref), _clean(cur)
  if len(r) == 0 or len(c) == 0: return np.nan
  return float(ks_2samp(r, c).statistic)

_val_m  = wallet_scores['wallet'].isin(val_wallets).values
_test_m = wallet_scores['wallet'].isin(test_wallets).values

model_score_map = {
  'baseline': wallet_scores['raw_score'].to_numpy(dtype=float),
  'if_only':  wallet_scores['if_score'].to_numpy(dtype=float),
  'tr_only':  wallet_scores['tr_score'].to_numpy(dtype=float),
}
if '_comp_ws' in globals() and isinstance(_comp_ws, pd.DataFrame):
  for col in ['lof_score', 'ocsvm_score', 'mlp_score']:
      if col in _comp_ws.columns:
          model_score_map[col.replace('_score', '')] = (
              _comp_ws[['wallet', col]].drop_duplicates('wallet')
              .set_index('wallet').reindex(wallet_scores['wallet'])
              [col].fillna(0.0).to_numpy(dtype=float)
          )

_base_val  = _clean(model_score_map['baseline'][_val_m])
_base_test = _clean(model_score_map['baseline'][_test_m])
_base_thr  = float(np.quantile(_base_val, 1 - ALPHA))
_base_fl   = (_base_test >= _base_thr).astype(int)

rows = []
for det, arr in model_score_map.items():
  vs = _clean(arr[_val_m]); ts = _clean(arr[_test_m])
  if not len(vs) or not len(ts): continue
  thr = float(np.quantile(vs, 1 - ALPHA))
  fl  = (ts >= thr).astype(int)
  inter = int(np.sum((fl == 1) & (_base_fl == 1)))
  union = int(np.sum((fl == 1) | (_base_fl == 1)))
  rows.append({
      'detector':        det,
      'flag_rate_%':     round(fl.mean() * 100, 2),
      'tail_contrast':   round(float(np.quantile(ts, 0.99) - np.quantile(ts, 0.50)), 4),
      'jaccard_base':    round(inter / union, 4) if union else 1.0,
      'rank_corr_base':  round(float(spearmanr(ts, _base_test).correlation), 4),
      'ks_val_test':     round(_drift_ks(vs, ts), 4),
  })

model_error_metrics = pd.DataFrame(rows)
_ord = {'baseline': 0, 'if_only': 1, 'tr_only': 2, 'lof': 3, 'ocsvm': 4, 'mlp': 5}
model_error_metrics = model_error_metrics.iloc[
  model_error_metrics['detector'].map(_ord).fillna(99).argsort()
].reset_index(drop=True)

# ── Seaborn heatmap ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

_plot = model_error_metrics.set_index('detector')
_norm = (_plot - _plot.min()) / (_plot.max() - _plot.min() + 1e-9)

fig, ax = plt.subplots(figsize=(10, max(3, len(_plot) * 0.7)))
sns.heatmap(
  _norm, annot=_plot.round(3), fmt='g',
  cmap='RdYlGn_r', linewidths=0.5, linecolor='white',
  cbar_kws={'label': 'Normalised (0=best, 1=worst)'},
  ax=ax,
)
ax.set_title('Model Error Metrics — No-Label Evaluation', fontsize=13, fontweight='bold')
ax.set_ylabel(''); ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('/workspace/checkpoints/model_error_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(model_error_metrics.to_string(index=False))

In [ ]:
# ── Pipeline evaluation (no ground truth) ────────────────────────────────────
# Covers: CP validity + p-value sharpness | AE MSE | IF separation | Jaccard stability
# MV/EM curves replaced by tail_contrast + Wasserstein (equivalent, no re-fits needed).

from scipy.stats import entropy as scipy_entropy

eval_rows = []

# ── 1. Conformal Prediction: empirical validity ───────────────────────────────
# Guarantee: realized false-alarm rate ≤ α on held-out normal (val) wallets.
_cp_val  = wallet_scores.loc[wallet_scores['wallet'].isin(val_wallets),  'conformal_p'].dropna().values
_cp_test = wallet_scores.loc[wallet_scores['wallet'].isin(test_wallets), 'conformal_p'].dropna().values

print('── CP Empirical Validity (realized false-alarm rate vs target α) ──')
print(f'  {"α_target":>10} {"α_realized":>12} {"gap_pp":>10} {"valid?":>8}')
for a in [0.005, 0.01, 0.02, 0.05]:
  realized = float(np.mean(_cp_val <= a)) if len(_cp_val) else np.nan
  gap = abs(realized - a) * 100 if np.isfinite(realized) else np.nan
  valid = '✓' if np.isfinite(gap) and gap < 1.0 else '✗'
  print(f'  {a:>10.3f} {realized:>12.4f} {gap:>9.2f}% {valid:>8}')
  eval_rows.append({'metric': f'CP_validity_alpha={a}', 'value': round(realized, 4) if np.isfinite(realized) else np.nan})

# ── 2. P-value sharpness (entropy) ───────────────────────────────────────────
if len(_cp_test) > 0:
  _hist, _ = np.histogram(_cp_test, bins=20, range=(0, 1), density=True)
  _hist = np.clip(_hist, 1e-9, None)
  _pval_entropy = float(scipy_entropy(_hist / _hist.sum()))
  _pval_low = float(np.mean(_cp_test < ALPHA))
  print(f'\n── P-value sharpness ──')
  print(f'  p-value entropy (lower=sharper): {_pval_entropy:.4f}')
  print(f'  fraction with p < α={ALPHA}:     {_pval_low:.4f}')
  eval_rows.append({'metric': 'pval_entropy',           'value': round(_pval_entropy, 4)})
  eval_rows.append({'metric': 'pval_flagged_at_alpha',  'value': round(_pval_low, 4)})

# ── 3. AE reconstruction MSE on test set ─────────────────────────────────────
if 'tr_raw' in globals():
  _test_tx_mask = df_flat['wallet'].isin(test_wallets).values
  _ae_mse_test  = float(np.mean(tr_raw[_test_tx_mask] ** 2))
  _ae_mse_train = float(np.mean(tr_raw[train_mask_tx] ** 2))
  _ae_ratio     = _ae_mse_test / max(_ae_mse_train, 1e-12)
  print(f'\n── AE Reconstruction MSE ──')
  print(f'  Train MSE : {_ae_mse_train:.6f}')
  print(f'  Test  MSE : {_ae_mse_test:.6f}')
  print(f'  Test/Train ratio (≈1 = no overfit): {_ae_ratio:.4f}')
  eval_rows.append({'metric': 'ae_mse_train',      'value': round(_ae_mse_train, 6)})
  eval_rows.append({'metric': 'ae_mse_test',       'value': round(_ae_mse_test,  6)})
  eval_rows.append({'metric': 'ae_mse_test/train', 'value': round(_ae_ratio,     4)})
else:
  print('\nAE MSE skipped: tr_raw not in scope (run AE scoring cell first).')

# ── 4. IF score separation (MV/EM proxy) ─────────────────────────────────────
if 'model_error_metrics' in globals() and len(model_error_metrics):
  _base_row = model_error_metrics[model_error_metrics['detector'] == 'baseline']
  if len(_base_row):
      _tc = float(_base_row['tail_contrast'].values[0])
      _ks = float(_base_row['ks_val_test'].values[0])
      print(f'\n── IF Score Separation (MV/EM proxy) ──')
      print(f'  Tail contrast  (p99-p50) : {_tc:.4f}  [higher = better separation]')
      print(f'  KS stat val→test         : {_ks:.4f}  [lower = more stable]')
      eval_rows.append({'metric': 'if_tail_contrast', 'value': round(_tc, 4)})
      eval_rows.append({'metric': 'if_ks_val_test',   'value': round(_ks, 4)})

# ── 5. Jaccard subsample stability ───────────────────────────────────────────
if 'struct_perm_df' in globals() and len(struct_perm_df):
  _jac_mean = float(struct_perm_df['jaccard_vs_original_flags'].mean())
  _jac_std  = float(struct_perm_df['jaccard_vs_original_flags'].std())
  print(f'\n── Jaccard Subsample Stability ──')
  print(f'  Mean Jaccard : {_jac_mean:.4f} ± {_jac_std:.4f}')
  print(f'  n_runs       : {len(struct_perm_df)}')
  eval_rows.append({'metric': 'jaccard_stability_mean', 'value': round(_jac_mean, 4)})
  eval_rows.append({'metric': 'jaccard_stability_std',  'value': round(_jac_std,  4)})
else:
  print('\nJaccard stability skipped: run permutation cell first.')

# ── Summary table ─────────────────────────────────────────────────────────────
eval_summary = pd.DataFrame(eval_rows)
print('\n── Evaluation Summary ──')
print(eval_summary.to_string(index=False))


In [ ]:
# ── 8. Export monitoring artifacts (CSV + PNG) ───────────────────────────────
import os

if 'no_label_dashboard' not in globals():
    raise RuntimeError('Run section 6 (No-label evaluation dashboard) first.')
if 'render_no_label_dashboard' not in globals():
    raise RuntimeError('Run section 7 first (dashboard renderer missing).')

EXPORT_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
OUT_DIR = os.path.join(EXPORT_ROOT, 'monitoring')
os.makedirs(OUT_DIR, exist_ok=True)

export_rows = []

# 1) Export dashboard tables
for name, obj in no_label_dashboard.items():
    if isinstance(obj, pd.DataFrame):
        out_path = os.path.join(OUT_DIR, f'{name}.csv')
        obj.to_csv(out_path, index=(name == 'agreement_corr'))
        export_rows.append({'artifact': name, 'type': 'csv', 'path': out_path, 'rows': int(len(obj))})

# 2) Render one compact dashboard figure via shared renderer
fig_path = os.path.join(OUT_DIR, 'no_label_dashboard.png')
_ = render_no_label_dashboard(no_label_dashboard, save_path=fig_path, show=False)
export_rows.append({'artifact': 'no_label_dashboard', 'type': 'png', 'path': fig_path, 'rows': np.nan})

# 3) Save manifest
manifest = pd.DataFrame(export_rows)
manifest_path = os.path.join(OUT_DIR, 'monitoring_manifest.csv')
manifest.to_csv(manifest_path, index=False)

print('Export complete. Artifacts:')
print(manifest.to_string(index=False))
print(f'\nOutput folder: {OUT_DIR}')

In [ ]:
# ── Main-report figures ───────────────────────────────────────────────────────
# Fig 1: Score separation (hist + KDE)
# Fig 2: Detector agreement scatter
# Fig 3: Conformal calibration curve
# Fig 4: Temporal monitoring trend

import os, matplotlib.pyplot as plt, seaborn as sns

MAIN_FIG_DIR = '/workspace/checkpoints/main_figures/'
os.makedirs(MAIN_FIG_DIR, exist_ok=True)

alpha_main   = float(ALPHA) if 'ALPHA' in globals() else 0.01
test_mask_m  = wallet_scores['wallet'].isin(test_wallets).values
val_mask_m   = wallet_scores['wallet'].isin(val_wallets).values
flag_m       = wallet_scores['conformal_flag'].fillna(0).astype(int).to_numpy()
n_tx_m       = wallet_scores['n_tx'].fillna(1).to_numpy(dtype=float) if 'n_tx' in wallet_scores.columns else np.ones(len(wallet_scores))

def _fin(a): return np.asarray(a, dtype=float)[np.isfinite(np.asarray(a, dtype=float))]

def _hist_kde(ax, data, color, label, alpha=0.35):
  x = _fin(data)
  if not len(x): return
  sns.histplot(x=x, bins=45, stat='density', alpha=alpha, color=color, label=f'{label} hist', ax=ax)
  if len(np.unique(x)) > 3:
      try: sns.kdeplot(x=x, color=color, linewidth=2.0, label=f'{label} KDE', ax=ax)
      except: pass

# ── Figure 1: Score separation ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=False)
for ax, (col, title) in zip(axes, [
  ('if_score',  'Isolation Forest score'),
  ('tr_score',  'Transformer AE score'),
  ('raw_score', 'Combined score'),
]):
  val_x = _fin(wallet_scores.loc[val_mask_m,  col].to_numpy(dtype=float))
  tst_x = _fin(wallet_scores.loc[test_mask_m, col].to_numpy(dtype=float))
  _hist_kde(ax, val_x, '#4C78A8', 'Val')
  _hist_kde(ax, tst_x, '#F58518', 'Test')
  if len(val_x):
      ax.axvline(np.quantile(val_x, 1 - alpha_main), linestyle='--', linewidth=1.6,
                 color='black', label=f'Threshold α={alpha_main:.3f}')
  ax.set_title(title); ax.set_xlabel(col); ax.grid(alpha=0.2)
axes[0].set_ylabel('Density')
axes[-1].legend(loc='best', fontsize=8)
fig.suptitle('Score separation: validation vs test', y=1.02)
plt.tight_layout()
fig.savefig(MAIN_FIG_DIR + '01_score_separation.png', dpi=170, bbox_inches='tight')
plt.show()
print('Saved: 01_score_separation.png')

# ── Figure 2: Detector agreement scatter ──────────────────────────────────────
plot_df = wallet_scores.loc[test_mask_m, ['wallet', 'if_score', 'tr_score']].copy()
plot_df['flag']        = flag_m[test_mask_m]
plot_df['flag_label']  = np.where(plot_df['flag'] == 1, 'Flagged', 'Unflagged')
plot_df['size_scaled'] = 20 + 90 * (np.log1p(n_tx_m[test_mask_m]) / np.log1p(max(n_tx_m.max(), 1)))

fig, ax = plt.subplots(figsize=(7.6, 6.2))
sns.scatterplot(data=plot_df, x='if_score', y='tr_score', hue='flag_label',
              size='size_scaled', sizes=(20, 110), alpha=0.70, ax=ax)
thr_if = float(np.quantile(_fin(wallet_scores.loc[val_mask_m, 'if_score']), 1 - alpha_main))
thr_tr = float(np.quantile(_fin(wallet_scores.loc[val_mask_m, 'tr_score']), 1 - alpha_main))
ax.axvline(thr_if, linestyle='--', linewidth=1.2, color='gray')
ax.axhline(thr_tr, linestyle='--', linewidth=1.2, color='gray')
ax.set_title('Detector agreement map (test wallets)')
ax.set_xlabel('IF score'); ax.set_ylabel('AE score'); ax.legend(loc='best')
plt.tight_layout()
fig.savefig(MAIN_FIG_DIR + '02_detector_agreement.png', dpi=170, bbox_inches='tight')
plt.show()
print('Saved: 02_detector_agreement.png')

# ── Figure 3: Conformal calibration curve ────────────────────────────────────
cal = _fin(wallet_scores.loc[val_mask_m,  'raw_score'].to_numpy(dtype=float))
tst = _fin(wallet_scores.loc[test_mask_m, 'raw_score'].to_numpy(dtype=float))
conf_rows = []
for a in [0.005, 0.01, 0.02, 0.05]:
  thr  = float(np.quantile(cal, 1 - a)) if len(cal) else np.nan
  rate = float((tst >= thr).mean() * 100) if len(tst) and np.isfinite(thr) else np.nan
  conf_rows.append({'target_pct': a * 100, 'realized_pct': rate})
conf_df = pd.DataFrame(conf_rows)

fig, ax = plt.subplots(figsize=(6.8, 5.8))
sns.lineplot(data=conf_df, x='target_pct', y='realized_pct', marker='o', linewidth=2.0, label='Realised', ax=ax)
_x = conf_df['target_pct'].to_numpy(dtype=float)
sns.lineplot(x=_x, y=_x, linestyle='--', linewidth=1.3, color='black', label='Ideal y=x', ax=ax)
ax.set_title('Conformal calibration (wallet-level)')
ax.set_xlabel('Target alpha (%)'); ax.set_ylabel('Realised flag rate (%)'); ax.legend()
plt.tight_layout()
fig.savefig(MAIN_FIG_DIR + '03_conformal_calibration.png', dpi=170, bbox_inches='tight')
plt.show()
print('Saved: 03_conformal_calibration.png')

# ── Figure 4: Temporal monitoring trend ──────────────────────────────────────
_ts_col = 'tr_tx_score' if 'tr_tx_score' in df_flat.columns else 'if_tx_score'
wallet_ts = (
  df_flat.loc[df_flat.groupby('wallet')[_ts_col].idxmax(), ['wallet', 'block_timestamp']]
  .drop_duplicates('wallet').set_index('wallet')['block_timestamp']
)
trend = wallet_scores.loc[test_mask_m, ['wallet', 'raw_score']].copy()
trend['flag']      = flag_m[test_mask_m]
trend['top_tx_ts'] = pd.to_datetime(trend['wallet'].map(wallet_ts), errors='coerce')
trend = trend.dropna(subset=['top_tx_ts'])

if len(trend):
  trend['week'] = trend['top_tx_ts'].dt.to_period('W').dt.start_time
  wf  = trend.groupby('week')['flag'].sum().sort_index()
  wrs = trend.groupby('week')['raw_score'].median().rolling(4, min_periods=1).mean().sort_index()

  fig, ax1 = plt.subplots(figsize=(10.2, 5.0))
  sns.lineplot(x=wf.index, y=wf.values, marker='o', linewidth=1.8, label='Flagged wallets/week', ax=ax1)
  ax1.set_xlabel('Week'); ax1.set_ylabel('Flagged wallets')
  ax2 = ax1.twinx()
  sns.lineplot(x=wrs.index, y=wrs.values, marker='s', linewidth=2.0,
               color='#E45756', label='Rolling median raw score (4w)', ax=ax2)
  ax2.set_ylabel('Raw score (median)')
  l1, lb1 = ax1.get_legend_handles_labels()
  l2, lb2 = ax2.get_legend_handles_labels()
  ax1.legend(l1 + l2, lb1 + lb2, loc='upper left')
  ax1.set_title('Temporal monitoring: weekly flagged volume and score trend')
  ax1.get_legend().remove() if ax1.get_legend() else None
  plt.tight_layout()
  fig.savefig(MAIN_FIG_DIR + '04_temporal_trend.png', dpi=170, bbox_inches='tight')
  plt.show()
  print('Saved: 04_temporal_trend.png')
else:
  print('Fig 4 skipped: no timestamp data available.')

print(f'\nAll figures saved to {MAIN_FIG_DIR}')

In [ ]:
# ── 12. Appendix visual pack (robustness + ops diagnostics) ─────────────────
import os
from scipy.stats import ks_2samp, wasserstein_distance

APP_FIG_DIR = '/workspace/monitoring/appendix_figures'
os.makedirs(APP_FIG_DIR, exist_ok=True)
appendix_manifest = []


def _finite_local(a):
  x = np.asarray(a, dtype=float)
  return x[np.isfinite(x)]


def _psi_month(expected, actual, bins=10, eps=1e-6):
  exp = _finite_local(expected)
  act = _finite_local(actual)
  if len(exp) == 0 or len(act) == 0:
      return np.nan
  edges = np.quantile(exp, np.linspace(0, 1, bins + 1))
  edges = np.unique(edges)
  if len(edges) < 3:
      return 0.0
  exp_hist, _ = np.histogram(exp, bins=edges)
  act_hist, _ = np.histogram(act, bins=edges)
  exp_pct = np.clip(exp_hist / max(exp_hist.sum(), 1), eps, None)
  act_pct = np.clip(act_hist / max(act_hist.sum(), 1), eps, None)
  return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def _wallet_time_map_appendix(df):
  if {'wallet', 'block_timestamp', 'combined_tx_score'}.issubset(df.columns):
      idx = df.groupby('wallet')['combined_tx_score'].idxmax()
  elif {'wallet', 'block_timestamp', 'tr_tx_score'}.issubset(df.columns):
      idx = df.groupby('wallet')['tr_tx_score'].idxmax()
  elif {'wallet', 'block_timestamp'}.issubset(df.columns):
      idx = df.groupby('wallet')['block_timestamp'].idxmax()
  else:
      return pd.Series(dtype='datetime64[ns]')
  out = (
      df.loc[idx, ['wallet', 'block_timestamp']]
      .drop_duplicates('wallet')
      .set_index('wallet')['block_timestamp']
  )
  return pd.to_datetime(out, errors='coerce')


# A1) Hyperparameter heatmaps (from section 3 sensitivity sweep)
if 'sensitivity_results' in globals() and isinstance(sensitivity_results, pd.DataFrame) and len(sensitivity_results):
  piv_flag = sensitivity_results.pivot(index='alpha', columns='w_if', values='flag_rate_pct').sort_index()
  piv_jac = sensitivity_results.pivot(index='alpha', columns='w_if', values='jaccard_vs_baseline').sort_index()

  fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))
  sns.heatmap(piv_flag, cmap='viridis', annot=True, fmt='.2f', cbar_kws={'label': 'Flag rate (%)'}, ax=axes[0])
  axes[0].set_title('Flag rate (%)')
  axes[0].set_xlabel('w_if'); axes[0].set_ylabel('alpha')

  sns.heatmap(piv_jac, cmap='magma', annot=True, fmt='.3f', cbar_kws={'label': 'Jaccard'}, ax=axes[1])
  axes[1].set_title('Jaccard vs baseline')
  axes[1].set_xlabel('w_if'); axes[1].set_ylabel('alpha')

  plt.tight_layout()
  a1_path = os.path.join(APP_FIG_DIR, 'A1_hyperparameter_heatmaps.png')
  fig.savefig(a1_path, dpi=170, bbox_inches='tight')
  plt.show()
  appendix_manifest.append({'figure': 'A1_hyperparameter_heatmaps', 'path': a1_path})
else:
  print('A1 skipped: sensitivity_results not found.')


# A2) IF contamination sweep curves (from section 9)
if 'if_sensitivity_results' in globals() and isinstance(if_sensitivity_results, pd.DataFrame) and len(if_sensitivity_results):
  d = if_sensitivity_results.sort_values('contamination').copy()
  d['jaccard_vs_ref_pct'] = 100.0 * d['jaccard_vs_ref'] if 'jaccard_vs_ref' in d.columns else np.nan
  d['rank_corr_vs_ref_pct'] = 100.0 * d['rank_corr_vs_ref'] if 'rank_corr_vs_ref' in d.columns else np.nan

  pcols = [c for c in ['flag_rate_pct', 'jaccard_vs_ref_pct', 'rank_corr_vs_ref_pct'] if c in d.columns]
  dm = d.melt(id_vars='contamination', value_vars=pcols, var_name='metric', value_name='percent')

  fig, ax = plt.subplots(figsize=(8.6, 5.0))
  sns.lineplot(data=dm, x='contamination', y='percent', hue='metric', marker='o', ax=ax)
  ax.set_xscale('log')
  ax.set_title('A2. IF contamination sensitivity')
  ax.set_xlabel('contamination'); ax.set_ylabel('Percent')
  ax.legend(loc='best')

  plt.tight_layout()
  a2_path = os.path.join(APP_FIG_DIR, 'A2_if_contamination_sensitivity.png')
  fig.savefig(a2_path, dpi=170, bbox_inches='tight')
  plt.show()
  appendix_manifest.append({'figure': 'A2_if_contamination_sensitivity', 'path': a2_path})
else:
  print('A2 skipped: if_sensitivity_results not found.')


# A4) Rolling drift panel by month (raw_score)
monthly_drift = pd.DataFrame()
if 'wallet_scores' in globals() and 'df_flat' in globals() and 'raw_score' in wallet_scores.columns:
  tmap = _wallet_time_map_appendix(df_flat)
  wd = wallet_scores[['wallet', 'raw_score']].copy()
  if 'test_wallets' in globals():
      wd = wd[wd['wallet'].isin(test_wallets)]
  wd['ts'] = pd.to_datetime(wd['wallet'].map(tmap), errors='coerce')
  wd = wd.dropna(subset=['ts'])

  if len(wd):
      wd['month'] = wd['ts'].dt.to_period('M').dt.to_timestamp()
      months = sorted(wd['month'].dropna().unique())

      rows = []
      prev = None
      for m in months:
          cur = _finite_local(wd.loc[wd['month'] == m, 'raw_score'].to_numpy(dtype=float))
          if prev is None:
              prev = cur; continue
          if len(prev) == 0 or len(cur) == 0:
              prev = cur; continue
          ks_stat, ks_p = ks_2samp(prev, cur)
          rows.append({
              'month': pd.Timestamp(m),
              'ks_stat': float(ks_stat),
              'ks_pvalue': float(ks_p),
              'wasserstein': float(wasserstein_distance(prev, cur)),
              'psi': float(_psi_month(prev, cur, bins=10)),
          })
          prev = cur

      monthly_drift = pd.DataFrame(rows)

      if len(monthly_drift):
          mdm = monthly_drift.melt(id_vars='month', value_vars=['ks_stat', 'wasserstein', 'psi'], var_name='metric', value_name='value')
          fig, ax = plt.subplots(figsize=(10.2, 4.8))
          sns.lineplot(data=mdm, x='month', y='value', hue='metric', marker='o', ax=ax)
          ax.set_title('A4. Rolling monthly drift (raw_score, month-to-month)')
          ax.set_xlabel('Month'); ax.set_ylabel('Drift metric value')
          ax.legend(loc='best')

          plt.tight_layout()
          a4_path = os.path.join(APP_FIG_DIR, 'A4_monthly_drift_panel.png')
          fig.savefig(a4_path, dpi=170, bbox_inches='tight')
          plt.show()
          appendix_manifest.append({'figure': 'A4_monthly_drift_panel', 'path': a4_path})


# A5) Alert budget curve
if 'raw_score' in wallet_scores.columns:
  if 'test_wallets' in globals():
      score_budget = _finite_local(wallet_scores.loc[wallet_scores['wallet'].isin(test_wallets), 'raw_score'].to_numpy(dtype=float))
  else:
      score_budget = _finite_local(wallet_scores['raw_score'].to_numpy(dtype=float))

  if len(score_budget):
      s = np.sort(score_budget)[::-1]
      k = np.arange(1, len(s) + 1)
      cum_mass_pct = 100.0 * np.cumsum(s) / (np.sum(s) + 1e-12)
      wallet_pct = 100.0 * k / len(s)

      curve_df = pd.DataFrame({'wallet_pct': wallet_pct, 'cum_mass_pct': cum_mass_pct})

      fig, ax = plt.subplots(figsize=(8.2, 5.2))
      sns.lineplot(data=curve_df, x='wallet_pct', y='cum_mass_pct', linewidth=2.2, label='Cumulative anomaly mass', ax=ax)
      sns.lineplot(data=curve_df, x='wallet_pct', y='wallet_pct', linestyle='--', linewidth=1.2, label='Uniform baseline', ax=ax)

      for p in [1, 5, 10]:
          idx_p = max(1, int(np.ceil((p / 100.0) * len(s)))) - 1
          _x = wallet_pct[idx_p]; _y = cum_mass_pct[idx_p]
          sns.scatterplot(x=[_x], y=[_y], s=40, color='black', ax=ax, legend=False)
          ax.text(_x, _y, f' Top {p}%: {_y:.1f}%', fontsize=8, va='bottom')

      ax.set_title('A5. Alert budget curve (top-K wallet concentration)')
      ax.set_xlabel('Top wallets reviewed (%)'); ax.set_ylabel('Captured anomaly-score mass (%)')
      ax.set_xlim(0, 100); ax.set_ylim(0, 100)
      ax.legend(loc='lower right')

      plt.tight_layout()
      a5_path = os.path.join(APP_FIG_DIR, 'A5_alert_budget_curve.png')
      fig.savefig(a5_path, dpi=170, bbox_inches='tight')
      plt.show()
      appendix_manifest.append({'figure': 'A5_alert_budget_curve', 'path': a5_path})

appendix_figures_manifest = pd.DataFrame(appendix_manifest)
print('Appendix figures generated:')
print(appendix_figures_manifest.to_string(index=False) if len(appendix_figures_manifest) else 'No figures generated.')
print(f'\nOutput folder: {APP_FIG_DIR}')

In [ ]:
import pickle
import numpy as np

np.save(CKPT + 'cal_scores.npy', cal_scores)

with open(CKPT + 'test_wallets.pkl', 'wb') as _f:
  pickle.dump(test_wallets, _f)

with open(CKPT + 'alpha.pkl', 'wb') as _f:
  pickle.dump(ALPHA, _f)

if 'decay_df' in globals() and len(decay_df):
  decay_df.to_parquet(CKPT + 'decay_df.parquet', index=False)
  print('Saved: cal_scores.npy, test_wallets.pkl, alpha.pkl, decay_df.parquet')
else:
  print('Saved: cal_scores.npy, test_wallets.pkl, alpha.pkl (decay_df not yet run)')

## Operational Context & Ethical Considerations

### Deployment assumptions
| Parameter | Value / Assumption |
|-----------|-------------------|
| Target FPR | ≤ α = 1 % (conformal guarantee, subject to exchangeability) |
| Minimum wallets needed to flag | 1 anomalous transaction (max aggregation) |
| Downstream action | Risk-tier assignment (low / medium / high / critical) |
| Human review required | Yes — no automated blocking |

### False-positive risk
- Flagging a wallet as anomalous has real-world consequences (account freezing,
  compliance investigations, reputational harm).
- At α = 0.01 the expected FPR is ≤ 1 % **only** if the exchangeability assumption
  holds.  The temporal KS test in the conformal validity section should be checked
  before each deployment.
- Any downstream action beyond risk-scoring **must** include a human review step.

### Label-free limitations
Because the system is fully unsupervised, there is no precision or recall estimate
against a ground-truth anomaly set.  The synthetic contamination test (Cell 15) is
a necessary proxy but not a substitute for a labelled evaluation:
- Detection rate on synthetic anomalies does **not** guarantee detection of novel
  real-world attack patterns.
- Consider enriching the evaluation with publicly available on-chain scam/exploit
  address lists (e.g., Solana Foundation reports, public MEV bot registries) as a
  held-out evaluation set without retraining.

### Scope
This model covers Solana mainnet transactions from September – November 2023.
It should **not** be applied to:
- Earlier periods without re-calibrating the conformal threshold.
- Other chains without retraining on chain-specific features.
- Real-time streaming without drift monitoring (see Cell 17 dashboard).

### References
- Tibshirani et al. (2019). *Conformal Prediction Under Covariate Shift.*
- Gibbs & Candès (2021). *Adaptive Conformal Inference Under Distribution Shift.*
  NeurIPS 2019.
- Liu et al. (2008). *Isolation Forest.*  ICDM 2008.
- Chalapathy & Chawla (2019). *Deep Learning for Anomaly Detection: A Survey.*
  arXiv:1901.03407.
